# Notebook 05: Urban Heat Island Hotspot Analysis

**Purpose:** Identify, delineate, and characterise persistent urban heat island hotspots across five Nigerian cities using complementary detection methods, validate results against stakeholder surveys, and map the underlying physiographic landscape composition of each hotspot. Extended stakeholder perception analysis links workshop responses to landscape character types across all 28 workshop cities.

**Study Year:** 2024 | **Cities (hotspot detection):** Abuja, Jos, Kano, Lagos, Port Harcourt | **Cities (stakeholder analysis):** All 28 workshop locations  
**Configuration:** `config.py` | **Functions:** `src/data_io.py`, `src/visualization.py`

---

## Methodology

### 1. Hotspot Detection

Two independent detection methods are applied to median LST rasters derived from the longest persistent heat event identified in Notebook 04:

**Percentile Threshold (P90)** -- Pixels exceeding the 90th percentile of the city-wide LST distribution. Captures statistical extremes without assuming spatial structure.

**Getis-Ord Gi\* Spatial Clustering** -- A local spatial autocorrelation statistic that identifies statistically significant clusters of high LST values (Ord & Getis, 1995). The statistic is defined as:

$$G_i^* = \frac{\sum_{j=1}^{n} w_{ij}x_j - \bar{X}\sum_{j=1}^{n} w_{ij}}{S\sqrt{\frac{n\sum_{j=1}^{n} w_{ij}^2 - \left(\sum_{j=1}^{n} w_{ij}\right)^2}{n-1}}}$$

where $w_{ij}$ are Queen-contiguity spatial weights, $x_j$ is the LST at location $j$, and $\bar{X}$, $S$ are the global mean and standard deviation. Z-scores exceeding 1.96 ($p < 0.05$) indicate statistically significant hot clusters.

**Persistence** -- Zones classified as hotspots in both day *and* night observations (Day $\cap$ Night), representing areas with continuous thermal stress and no nocturnal cooling relief.

### 2. KDE Delineation

Binary hotspot masks are converted to smooth polygon boundaries via Kernel Density Estimation. Because the native 1 km grid provides insufficient point density for reliable KDE, masks are first resampled to 250 m (nearest-neighbour) in memory. Scott's rule determines the bandwidth:

$$h = n^{-1/(d+4)}\,\sigma$$

where $n$ is the number of hotspot points, $d = 2$, and $\sigma$ is the coordinate standard deviation.

Three nested contour levels define hotspot criticality:
- **90 %** -- Core hotspot (top 10 % density, highest priority)
- **75 %** -- Moderately critical zone
- **50 %** -- Broader fringe, lower priority

Isolated hotspots with fewer than 10 pixels are delineated using 500 m circular buffers instead of KDE.

### 3. Attribute Enrichment

Each delineated polygon is enriched with:
- **Landscape character** (dominant Cluster25 class, climate, geology, elevation) via spatial overlay with a 25-class physiographic classification of Nigeria
- **LST statistics** (min, mean, max, std, median, pixel count) for day and night rasters
- **Population exposure** (daytime and nighttime counts from GHSL-POP 2020) via zonal statistics

### 4. Stakeholder Validation

Hotspot polygons are validated against stakeholder-reported heat-stress locations collected in Notebook 02. A simple polygon-containment check (no buffer) yields a **hit rate** per city and period, measuring the proportion of stakeholder hotspots captured by any detection method.

### 5. Visualization

Modularised plotting functions in `src/visualization.py` produce six categories of outputs:
- **Section 6.1** -- Per-city grid maps (Day | Night | Persistent) showing P90 and Gi\* polygons on Cluster25 landscape base
- **Section 6.2** -- Gi\* Z-score distribution histograms (Day and Night) per city
- **Section 6.3** -- Stacked bar charts of hotspot area by contour level, method, and Cluster25 composition
- **Section 6.4** -- Half-donut cross-section charts of population exposure by period, method, and contour level (one per city)
- **Section 6.5** -- National summary map showing all 28 workshop cities with proportional symbols for population exposure
- **Section 6.6** -- Stakeholder perception charts: reasons/mitigation by landscape type, and tree species heatmaps (4 variants)

All outputs include skip logic: regeneration is bypassed when files already exist with unchanged parameters.

---

## Notebook Structure

| Section | Content |
|---------|---------|
| 1 | Setup, data loading (LST rasters, landscape, survey) |
| 2 | Hotspot identification (P90, Gi\*, persistence) |
| 3 | KDE delineation and polygon generation |
| 4 | Attribute enrichment and multi-CRS export |
| 5 | Stakeholder validation (hit-rate analysis) |
| 6.1–6.3 | Hotspot visualisation (maps, distributions, area charts) |
| 6.4 | Population exposure half-donut cross-sections |
| 6.5 | National summary map (all cities on Nigeria landscape) |
| 6.6 | Stakeholder perception by landscape character (reasons, mitigation, tree species) |

---

**References**

- Ord, J.K. and Getis, A. (1995) 'Local spatial autocorrelation statistics: distributional issues and an application', *Geographical Analysis*, 27(4), pp. 286--306.
- Scott, D.W. (1992) *Multivariate Density Estimation: Theory, Practice, and Visualization*. Wiley.
- Schiavina, M. et al. (2023) 'GHS-POP R2023A', European Commission, JRC.

## Section 1: Setup and Data Loading

### 1.1 Environment Setup

Import libraries and custom modules, load project configuration from `config.py`.

In [ ]:
# Import Standard Libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from pathlib import Path
from datetime import datetime
import json

# Spatial Analysis Libraries
import rasterio
from rasterio.features import rasterize
from rasterio.mask import mask
from rasterio.plot import show
from scipy.stats import gaussian_kde
from scipy import ndimage
from skimage import measure
from shapely.geometry import Polygon, MultiPolygon, Point, box
from shapely.ops import unary_union

# PySAL spatial analysis (modern imports)
from libpysal import weights
from esda import Moran, Moran_Local, G_Local

# Import Custom Modules
import sys
sys.path.append(str(Path.cwd().parent))  # For config.py in project root
sys.path.append(str(Path.cwd().parent / 'src'))  # For src modules

# Reload config to get latest changes
import importlib
import config
importlib.reload(config)

from config import (
    STUDY_YEAR, HOTSPOT_CONFIG, KDE_CONFIG, LANDSCAPE_ASSESSMENT,
    GHSL_AVAILABLE_YEARS, get_closest_ghsl_year,
    load_cities_from_daily_analysis, verify_ghsl_availability,
    OUTPUT_DIR, GEE_CONFIG,
    LANDSCAPE_DATA_PATH, LANDSCAPE_CLASS_ATTRIBUTES, LANDSCAPE_CLUSTER_ATTRIBUTE,
    LST_RASTER_CONFIG, HOTSPOT_EXPORT_CONFIG,
    GHSL_POP_CONFIG
)
# Import available visualization functions (others will be added as needed)
import visualization
import data_io
import temporal_analysis_utils

# Alias for convenience
OUTPUT_BASE_PATH = OUTPUT_DIR

# Configure Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10

# Display Configuration Summary
print("="*70)
print("NOTEBOOK 05: URBAN HEAT ISLAND HOTSPOT ANALYSIS")
print("="*70)

# Load cities from Notebook 04 (Daily Analysis outputs)
print("\n[1/3] Loading Cities from Daily Analysis...")
CITIES = load_cities_from_daily_analysis(STUDY_YEAR)
if not CITIES:
    raise RuntimeError("No cities found! Please run Notebook 04 first.")
print(f"✓ Loaded {len(CITIES)} cities: {', '.join(CITIES)}")

# Verify GHSL Population data availability
print("\n[2/3] Verifying GHSL Population Data Availability...")
ghsl_status = verify_ghsl_availability()
if not ghsl_status['gee_available']:
    print(f"\n⚠ Warning: GEE not available. Population analysis will be skipped.")
    print(f"  Reason: {ghsl_status['status_message']}")
else:
    print(f"✓ GEE available with {len(ghsl_status['available_years'])} GHSL years")

# Get closest GHSL year with detailed output
print("\n[3/3] Selecting GHSL Population Year...")
ghsl_year = get_closest_ghsl_year(STUDY_YEAR, verbose=True)

# Final Configuration Summary
print("\n" + "="*70)
print("CONFIGURATION SUMMARY")
print("="*70)
print(f"Study Year: {STUDY_YEAR}")
print(f"Cities Analyzed: {len(CITIES)}")
print(f"Cities: {', '.join(CITIES)}")
print(f"\nGHSL Population Year: {ghsl_year}")
print(f"GHSL Status: {'✓ Available' if ghsl_status['gee_available'] else '✗ Unavailable'}")
print(f"\nHotspot Detection Methods:")
print(f"  - Percentile Threshold: P{HOTSPOT_CONFIG['percentile_hot']}")
print(f"  - Getis-Ord Gi* (p < {HOTSPOT_CONFIG['gi_star_alpha']})")
print(f"  - Method Validation: P90 ∩ Gi*")
print(f"  - Persistence: Day ∩ Night")
print(f"\nKDE Configuration:")
print(f"  - Bandwidth Method: {KDE_CONFIG['bandwidth_method']}")
print(f"  - Contour Levels: {KDE_CONFIG['contour_levels']}")
print(f"\nDate/Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print("\n✓ Environment setup complete.\n")

### 1.2 Load Persistent Heat Rasters

Load day and night median-LST rasters produced by Notebook 04 for each city. These rasters correspond to the longest consecutive persistent heat event identified during the study year.

In [ ]:
print("="*70)
print("📥 LOADING PERSISTENT HEAT RASTERS FROM NOTEBOOK 04")
print("="*70)

# Initialize storage dictionary for raster data
persistent_heat_rasters = {}

# Base path to SEVIRI outputs
seviri_base_path = OUTPUT_BASE_PATH / "04_trend_analysis_daily" / "SEVIRI_LST"

print(f"\n🔍 Scanning SEVIRI outputs: {seviri_base_path}")
print(f"   Looking for cities: {', '.join(CITIES)}\n")

# Load rasters for each city
for city in CITIES:
    city_dir = seviri_base_path / f"City_{city}"
    
    if not city_dir.exists():
        print(f"  ⚠️  {city}: Directory not found")
        continue
    
    # Find clipped raster files (WGS84 1km, clipped to AOI)
    day_pattern = f"{city}_SEVIRI_day_*_median_WGS84_1km_clipped.tif"
    night_pattern = f"{city}_SEVIRI_night_*_median_WGS84_1km_clipped.tif"
    
    day_files = list(city_dir.glob(day_pattern))
    night_files = list(city_dir.glob(night_pattern))
    
    if not day_files or not night_files:
        print(f"  ⚠️  {city}: Rasters not found (day={len(day_files)}, night={len(night_files)})")
        continue
    
    # Take first match (should be only one per city)
    day_raster = day_files[0]
    night_raster = night_files[0]
    
    # Load metadata if available
    metadata_filename = day_raster.name.replace("_clipped.tif", "").replace("WGS84_1km", "MSG3km") + "_metadata.json"
    metadata_file = city_dir / metadata_filename
    metadata = {}
    if metadata_file.exists():
        with open(metadata_file, 'r') as f:
            metadata = json.load(f)
    
    # Load rasters with rasterio
    try:
        with rasterio.open(day_raster) as src:
            day_data = src.read(1)
            day_transform = src.transform
            day_crs = src.crs
            day_bounds = src.bounds
        
        with rasterio.open(night_raster) as src:
            night_data = src.read(1)
            night_transform = src.transform
            night_crs = src.crs
            night_bounds = src.bounds
        
        # Store in dictionary
        persistent_heat_rasters[city] = {
            'day': {
                'data': day_data,
                'transform': day_transform,
                'crs': day_crs,
                'bounds': day_bounds,
                'file': day_raster,
                'metadata': metadata
            },
            'night': {
                'data': night_data,
                'transform': night_transform,
                'crs': night_crs,
                'bounds': night_bounds,
                'file': night_raster,
                'metadata': {}  # Night metadata separate
            }
        }
        
        # Get event dates from metadata
        day_event = f"{metadata.get('event_start_date', 'N/A')} to {metadata.get('event_end_date', 'N/A')}"
        day_duration = metadata.get('event_duration_days', 'N/A')
        
        print(f"  ✓  {city}:")
        print(f"      Day event:  {day_event} ({day_duration} days)")
        print(f"      Raster shape: {day_data.shape} (1km resolution)")
        print(f"      Day LST range:   {day_data[day_data > 0].min():.1f} - {day_data[day_data > 0].max():.1f} °C")
        print(f"      Night LST range: {night_data[night_data > 0].min():.1f} - {night_data[night_data > 0].max():.1f} °C")
        
    except Exception as e:
        print(f"  ✗  {city}: Error loading rasters - {str(e)}")
        continue

print(f"\n{'='*70}")
print(f"✅ Successfully loaded rasters for {len(persistent_heat_rasters)} cities")
print(f"{'='*70}\n")

## Section 2: Hotspot Identification

Apply P90 percentile thresholds and Getis-Ord Gi\* statistics to identify hotspot pixels for day and night periods, then derive persistence masks (Day $\cap$ Night) and a summary table across all cities.

In [ ]:
print("="*70)
print("📊 SECTION 2.1: P90 THRESHOLD ANALYSIS")
print("="*70)

# Initialize storage for hotspot results
hotspot_results = {}

# P90 threshold from config
percentile = HOTSPOT_CONFIG['percentile_hot']

print(f"\nComputing P{percentile} thresholds for each city...\n")

for city in CITIES:
    print(f"🏙️  {city}")
    print("-" * 50)
    
    # Get raster data
    day_data = persistent_heat_rasters[city]['day']['data']
    night_data = persistent_heat_rasters[city]['night']['data']
    
    # Mask out nodata (assuming 0 or negative values are nodata)
    day_valid = day_data[day_data > 0]
    night_valid = night_data[night_data > 0]
    
    # Calculate P90 thresholds
    day_p90 = np.percentile(day_valid, percentile)
    night_p90 = np.percentile(night_valid, percentile)
    
    # Generate binary masks (True = hotspot)
    day_p90_mask = (day_data > 0) & (day_data >= day_p90)
    night_p90_mask = (night_data > 0) & (night_data >= night_p90)
    
    # Count hotspot pixels
    day_p90_count = day_p90_mask.sum()
    night_p90_count = night_p90_mask.sum()
    
    # Calculate areas (1km resolution = 1 km²/pixel)
    day_p90_area = day_p90_count * 1.0  # km²
    night_p90_area = night_p90_count * 1.0  # km²
    
    # Calculate percentage of valid area
    total_valid_pixels = (day_data > 0).sum()
    day_p90_pct = (day_p90_count / total_valid_pixels * 100) if total_valid_pixels > 0 else 0
    night_p90_pct = (night_p90_count / total_valid_pixels * 100) if total_valid_pixels > 0 else 0
    
    # Store results
    hotspot_results[city] = {
        'day': {
            'p90_threshold': day_p90,
            'p90_mask': day_p90_mask,
            'p90_count': day_p90_count,
            'p90_area_km2': day_p90_area,
            'p90_pct': day_p90_pct
        },
        'night': {
            'p90_threshold': night_p90,
            'p90_mask': night_p90_mask,
            'p90_count': night_p90_count,
            'p90_area_km2': night_p90_area,
            'p90_pct': night_p90_pct
        }
    }
    
    # Display results
    print(f"  Day P{percentile}:   {day_p90:.2f} °C")
    print(f"    Hotspot pixels: {day_p90_count:,} ({day_p90_pct:.1f}% of city)")
    print(f"    Hotspot area:   {day_p90_area:.1f} km²")
    
    print(f"  Night P{percentile}: {night_p90:.2f} °C")
    print(f"    Hotspot pixels: {night_p90_count:,} ({night_p90_pct:.1f}% of city)")
    print(f"    Hotspot area:   {night_p90_area:.1f} km²")
    print()

print("="*70)
print(f"✅ P{percentile} thresholds computed for all cities")
print("="*70 + "\n")

In [ ]:
print("="*70)
print("🔬 SECTION 2.2: GETIS-ORD Gi* SPATIAL CLUSTERING")
print("="*70)

# Get significance threshold from config
sig_level = HOTSPOT_CONFIG['gi_star_alpha']
z_threshold = 1.96  # Two-tailed test at p < 0.05

print(f"\nSpatial weights: Queen contiguity (8 neighbors)")
print(f"Significance level: p < {sig_level} (Z > {z_threshold})\n")

for city in CITIES:
    print(f"🏙️  {city}")
    print("-" * 50)
    
    # Process day and night separately
    for period in ['day', 'night']:
        data = persistent_heat_rasters[city][period]['data']
        
        # Create coordinate grid for valid pixels
        rows, cols = data.shape
        valid_mask = data > 0
        
        # Get row, col indices of valid pixels
        row_indices, col_indices = np.where(valid_mask)
        valid_data = data[valid_mask]
        n_valid = len(valid_data)
        
        if n_valid < 10:
            print(f"  {period.capitalize()}: Insufficient data (n={n_valid})")
            continue
        
        # Create DataFrame for spatial analysis
        coords = np.column_stack([row_indices, col_indices])
        gdf = gpd.GeoDataFrame({
            'temp': valid_data,
            'row': row_indices,
            'col': col_indices
        }, geometry=[Point(c, r) for c, r in coords])
        
        # Build queen contiguity weights (8-neighbors: cardinal + diagonal)
        w = weights.contiguity.Queen.from_dataframe(gdf, use_index=False)
        
        # Compute Getis-Ord Gi* using G_Local
        gi_star = G_Local(gdf['temp'].values, w, star=True)
        
        # Identify hotspots (positive Z-score, statistically significant)
        hotspot_mask_flat = (gi_star.Zs > z_threshold)
        
        # Create full raster mask
        hotspot_mask_full = np.zeros(data.shape, dtype=bool)
        hotspot_mask_full[row_indices[hotspot_mask_flat], col_indices[hotspot_mask_flat]] = True
        
        # Count hotspot pixels
        gi_star_count = hotspot_mask_flat.sum()
        gi_star_area = gi_star_count * 1.0  # km²
        gi_star_pct = (gi_star_count / n_valid * 100) if n_valid > 0 else 0
        
        # Store in results
        if period not in hotspot_results[city]:
            hotspot_results[city][period] = {}
        
        hotspot_results[city][period]['gi_star_z'] = gi_star.Zs
        hotspot_results[city][period]['gi_star_mask'] = hotspot_mask_full
        hotspot_results[city][period]['gi_star_count'] = gi_star_count
        hotspot_results[city][period]['gi_star_area_km2'] = gi_star_area
        hotspot_results[city][period]['gi_star_pct'] = gi_star_pct
        
        # Display results
        print(f"  {period.capitalize()} Gi*:")
        print(f"    Valid pixels analyzed: {n_valid:,}")
        print(f"    Hotspot pixels: {gi_star_count:,} ({gi_star_pct:.1f}% of valid)")
        print(f"    Hotspot area: {gi_star_area:.1f} km²")
        
        # Only show statistics if hotspots exist
        if gi_star_count > 0:
            print(f"    Mean Z-score (hotspots): {gi_star.Zs[hotspot_mask_flat].mean():.2f}")
            print(f"    Max Z-score (hotspots): {gi_star.Zs[hotspot_mask_flat].max():.2f}")
        else:
            print(f"    No significant hotspots detected (Z > {z_threshold})")
    
    print()

print("="*70)
print("✅ Gi* spatial clustering completed for all cities")
print("="*70 + "\n")

In [ ]:
print("="*70)
print("✅ SECTION 2.3: VALIDATED HOTSPOTS (P90 ∩ Gi*)")
print("="*70)

print(f"\nIntersecting P{percentile} and Gi* hotspots...\n")

for city in CITIES:
    print(f"🏙️  {city}")
    print("-" * 50)
    
    for period in ['day', 'night']:
        # Get P90 and Gi* masks
        p90_mask = hotspot_results[city][period]['p90_mask']
        gi_star_mask = hotspot_results[city][period]['gi_star_mask']
        
        # Intersection: both P90 AND Gi*
        validated_mask = p90_mask & gi_star_mask
        
        # Count validated hotspots
        validated_count = validated_mask.sum()
        validated_area = validated_count * 1.0  # km²
        
        # Calculate overlap statistics
        p90_count = hotspot_results[city][period]['p90_count']
        gi_star_count = hotspot_results[city][period]['gi_star_count']
        
        # Precision: % of P90 hotspots confirmed by Gi*
        precision = (validated_count / p90_count * 100) if p90_count > 0 else 0
        
        # Recall: % of Gi* hotspots confirmed by P90
        recall = (validated_count / gi_star_count * 100) if gi_star_count > 0 else 0
        
        # F1-Score
        f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        # Store results
        hotspot_results[city][period]['validated_mask'] = validated_mask
        hotspot_results[city][period]['validated_count'] = validated_count
        hotspot_results[city][period]['validated_area_km2'] = validated_area
        hotspot_results[city][period]['validation_precision'] = precision
        hotspot_results[city][period]['validation_recall'] = recall
        hotspot_results[city][period]['validation_f1'] = f1_score
        
        # Display results
        print(f"  {period.capitalize()}:")
        print(f"    P90 hotspots:       {p90_count:,}")
        print(f"    Gi* hotspots:       {gi_star_count:,}")
        print(f"    Validated (∩):      {validated_count:,} ({validated_area:.1f} km²)")
        print(f"    Precision:          {precision:.1f}%")
        print(f"    Recall:             {recall:.1f}%")
        print(f"    F1-Score:           {f1_score:.1f}%")
    
    print()

print("="*70)
print("✅ Validated hotspots generated")
print("="*70 + "\n")

In [ ]:
print("="*70)
print("🌓 SECTION 2.4: PERSISTENT HEAT ZONES (Day ∩ Night)")
print("="*70)

print("\nIdentifying zones hot in BOTH day and night (24-hour thermal stress)...\n")

for city in CITIES:
    print(f"🏙️  {city}")
    print("-" * 50)
    
    # Initialize persistent storage
    hotspot_results[city]['persistent'] = {}
    
    # P90 Persistent: Day ∩ Night
    p90_day = hotspot_results[city]['day']['p90_mask']
    p90_night = hotspot_results[city]['night']['p90_mask']
    p90_persistent = p90_day & p90_night
    p90_persistent_count = p90_persistent.sum()
    p90_persistent_area = p90_persistent_count * 1.0
    
    # Gi* Persistent: Day ∩ Night
    gi_star_day = hotspot_results[city]['day']['gi_star_mask']
    gi_star_night = hotspot_results[city]['night']['gi_star_mask']
    gi_star_persistent = gi_star_day & gi_star_night
    gi_star_persistent_count = gi_star_persistent.sum()
    gi_star_persistent_area = gi_star_persistent_count * 1.0
    
    # Validated Persistent: Day ∩ Night
    validated_day = hotspot_results[city]['day']['validated_mask']
    validated_night = hotspot_results[city]['night']['validated_mask']
    validated_persistent = validated_day & validated_night
    validated_persistent_count = validated_persistent.sum()
    validated_persistent_area = validated_persistent_count * 1.0
    
    # Store persistent results
    hotspot_results[city]['persistent'] = {
        'p90_mask': p90_persistent,
        'p90_count': p90_persistent_count,
        'p90_area_km2': p90_persistent_area,
        
        'gi_star_mask': gi_star_persistent,
        'gi_star_count': gi_star_persistent_count,
        'gi_star_area_km2': gi_star_persistent_area,
        
        'validated_mask': validated_persistent,
        'validated_count': validated_persistent_count,
        'validated_area_km2': validated_persistent_area
    }
    
    # Display results
    print(f"  P{percentile} Persistent:")
    print(f"    Day only:     {(p90_day & ~p90_night).sum():,} pixels")
    print(f"    Night only:   {(p90_night & ~p90_day).sum():,} pixels")
    print(f"    Day ∩ Night:  {p90_persistent_count:,} pixels ({p90_persistent_area:.1f} km²)")
    
    print(f"  Gi* Persistent:")
    print(f"    Day only:     {(gi_star_day & ~gi_star_night).sum():,} pixels")
    print(f"    Night only:   {(gi_star_night & ~gi_star_day).sum():,} pixels")
    print(f"    Day ∩ Night:  {gi_star_persistent_count:,} pixels ({gi_star_persistent_area:.1f} km²)")
    
    print(f"  Validated Persistent:")
    print(f"    Day only:     {(validated_day & ~validated_night).sum():,} pixels")
    print(f"    Night only:   {(validated_night & ~validated_day).sum():,} pixels")
    print(f"    Day ∩ Night:  {validated_persistent_count:,} pixels ({validated_persistent_area:.1f} km²)")
    
    print()

print("="*70)
print("✅ Persistent heat zones identified")
print("="*70 + "\n")

In [ ]:
print("="*70)
print("📋 SECTION 2.5: HOTSPOT DETECTION SUMMARY")
print("="*70 + "\n")

# Build comprehensive summary table
summary_rows = []

for city in CITIES:
    for method in ['P90', 'Gi*', 'Validated']:
        for period in ['Day', 'Night', 'Persistent']:
            # Map method names to result keys
            method_key = method.lower().replace('*', 'star')
            if method == 'P90':
                method_key = 'p90'
            elif method == 'Gi*':
                method_key = 'gistar'
            elif method == 'Validated':
                method_key = 'validated'
            
            # Get period data
            if period == 'Persistent':
                period_data = hotspot_results[city]['persistent']
            else:
                period_data = hotspot_results[city][period.lower()]
            
            # Get metrics
            count = period_data.get(f'{method_key}_count', 0)
            area = period_data.get(f'{method_key}_area_km2', 0)
            
            # Calculate % of city
            total_pixels = (persistent_heat_rasters[city]['day']['data'] > 0).sum()
            pct_city = (count / total_pixels * 100) if total_pixels > 0 else 0
            
            # Get LST statistics for hotspot pixels
            if count > 0:
                mask = period_data.get(f'{method_key}_mask', None)
                if mask is not None and mask.any():
                    if period == 'Persistent':
                        # Use day LST for persistent hotspots
                        lst_data = persistent_heat_rasters[city]['day']['data']
                    else:
                        lst_data = persistent_heat_rasters[city][period.lower()]['data']
                    
                    hotspot_lst = lst_data[mask]
                    lst_min = hotspot_lst.min()
                    lst_max = hotspot_lst.max()
                    lst_mean = hotspot_lst.mean()
                    lst_std = hotspot_lst.std()
                else:
                    lst_min = lst_max = lst_mean = lst_std = np.nan
            else:
                lst_min = lst_max = lst_mean = lst_std = np.nan
            
            summary_rows.append({
                'City': city,
                'Method': method,
                'Period': period,
                'Pixels': count,
                'Area_km2': area,
                'Pct_City': pct_city,
                'LST_Min_K': lst_min,
                'LST_Max_K': lst_max,
                'LST_Mean_K': lst_mean,
                'LST_Std_K': lst_std
            })

# Create DataFrame
summary_df = pd.DataFrame(summary_rows)

# Display formatted table
print("Hotspot Detection Summary (All Cities)")
print("="*120)
print(summary_df.to_string(index=False))
print("="*120 + "\n")

# Summary statistics by method
print("Aggregate Statistics by Method:")
print("-"*70)
method_summary = summary_df.groupby('Method').agg({
    'Pixels': ['sum', 'mean'],
    'Area_km2': ['sum', 'mean'],
    'Pct_City': 'mean'
}).round(2)
print(method_summary)
print()

# Summary statistics by period
print("Aggregate Statistics by Period:")
print("-"*70)
period_summary = summary_df.groupby('Period').agg({
    'Pixels': ['sum', 'mean'],
    'Area_km2': ['sum', 'mean'],
    'Pct_City': 'mean'
}).round(2)
print(period_summary)
print()

print("="*70)
print("✅ Section 2: Hotspot Identification Complete")
print("="*70 + "\n")

# Store summary table for later use
hotspot_summary_df = summary_df

## Section 3: KDE Delineation and Polygon Generation

Convert binary hotspot masks into smooth polygon boundaries using Kernel Density Estimation. Masks are resampled to 250 m in memory to provide adequate point density, then discarded after polygon extraction.

### 3.1 Resample Binary Masks and Extract Coordinates

A two-tier strategy ensures all hotspots are delineated regardless of size:

| Hotspot size | Method | Rationale |
|---|---|---|
| >= 10 pixels | KDE path: resample to 250 m, fit KDE, extract contours | Sufficient spatial distribution for density estimation |
| < 10 pixels | Buffer path: 500 m circular buffers, merge overlapping | Too few points for reliable KDE |

In [ ]:
print("="*70)
print("📐 SECTION 3.1: TWO-TIER HOTSPOT DELINEATION")
print("="*70)

# Import required libraries
from scipy.ndimage import zoom

# Reload config to get RESAMPLE_CONFIG
importlib.reload(config)
from config import RESAMPLE_CONFIG

# Get resampling parameters
resample_resolution_m = RESAMPLE_CONFIG['resolution_m']
resample_method = RESAMPLE_CONFIG['method']
resample_order = RESAMPLE_CONFIG['order']
min_pixels = RESAMPLE_CONFIG['min_pixels_for_kde']

# Buffer configuration for isolated hotspots
buffer_radius_km = 0.5  # 500 meters

# Calculate resample factor (1km → target resolution)
original_resolution_m = 1000  # SEVIRI 1km
resample_factor = original_resolution_m / resample_resolution_m

print(f"\n{'='*60}")
print("TWO-TIER DELINEATION STRATEGY")
print(f"{'='*60}")
print(f"\n📊 KDE PATH (≥{min_pixels} pixels):")
print(f"  • Original resolution: {original_resolution_m}m")
print(f"  • Target resolution:   {resample_resolution_m}m")
print(f"  • Resample factor:     {resample_factor}x")
print(f"  • Interpolation:       {resample_method} (order={resample_order})")
print(f"  • Output:              Coordinates for KDE (Section 3.2)")

print(f"\n🎯 BUFFER PATH (<{min_pixels} pixels):")
print(f"  • Buffer radius:       {buffer_radius_km*1000:.0f}m")
print(f"  • Method:              Circular buffers + merge overlapping")
print(f"  • Output:              Direct polygon generation")
print(f"{'='*60}\n")

# Storage for results
hotspot_coordinates = {}  # For KDE path
isolated_hotspot_polygons = {}  # For buffer path
resampling_stats = []
buffer_stats = []

# Import tqdm for progress tracking
from tqdm.auto import tqdm

# Process each city
for city in tqdm(CITIES, desc="Cities", unit="city"):
    print(f"\n🏙️  {city}")
    print("-" * 60)
    
    # Get affine transform for coordinate conversion
    transform_1km = persistent_heat_rasters[city]['day']['transform']
    crs = persistent_heat_rasters[city]['day']['crs']
    
    # Calculate transform for resampled resolution
    from affine import Affine
    transform_resampled = transform_1km * Affine.scale(1/resample_factor)
    
    hotspot_coordinates[city] = {}
    isolated_hotspot_polygons[city] = {}
    
    # Process each method × period combination
    for method in ['p90', 'gi_star', 'validated']:
        for period in ['day', 'night', 'persistent']:
            
            # Get binary mask from Section 2 results
            if period == 'persistent':
                period_data = hotspot_results[city]['persistent']
            else:
                period_data = hotspot_results[city][period]
            
            mask_key = f'{method}_mask'
            if mask_key not in period_data:
                continue
            
            mask_1km = period_data[mask_key]
            n_pixels_original = mask_1km.sum()
            
            # Skip if no hotspots
            if n_pixels_original == 0:
                print(f"  {method.upper():10s} {period.capitalize():10s}: No hotspots detected")
                continue
            
            key = f"{method}_{period}"
            
            # ========================================
            # PATH 1: KDE (≥10 pixels)
            # ========================================
            if n_pixels_original >= min_pixels:
                # Resample binary mask to finer resolution (in-memory)
                mask_resampled = zoom(mask_1km.astype(float), zoom=resample_factor, order=resample_order)
                mask_resampled = mask_resampled > 0.5  # Convert back to binary
                n_pixels_resampled = mask_resampled.sum()
                
                # Extract row, col indices of True pixels
                row_indices, col_indices = np.where(mask_resampled)
                
                # Convert to geographic coordinates (lat, lon)
                coords_list = []
                for row, col in zip(row_indices, col_indices):
                    lon, lat = transform_resampled * (col + 0.5, row + 0.5)  # Pixel center
                    coords_list.append([lon, lat])
                
                coords_array = np.array(coords_list)
                
                # Store coordinates for KDE
                hotspot_coordinates[city][key] = {
                    'coordinates': coords_array,
                    'n_pixels_original': n_pixels_original,
                    'n_pixels_resampled': n_pixels_resampled,
                    'transform': transform_resampled,
                    'method': method,
                    'period': period,
                    'delineation_path': 'KDE'
                }
                
                # Track statistics
                resampling_stats.append({
                    'City': city,
                    'Method': method.upper(),
                    'Period': period.capitalize(),
                    'Original_Pixels': n_pixels_original,
                    'Resampled_Pixels': n_pixels_resampled,
                    'Path': 'KDE'
                })
                
                print(f"  {method.upper():10s} {period.capitalize():10s}: KDE path → {n_pixels_original:4d} → {n_pixels_resampled:6d} pixels")
                
                # Free memory immediately
                del mask_resampled
            
            # ========================================
            # PATH 2: BUFFER (<10 pixels)
            # ========================================
            else:
                # Extract pixel coordinates at 1km resolution
                row_indices, col_indices = np.where(mask_1km)
                
                # Create buffers around each pixel
                polygons = []
                for row, col in zip(row_indices, col_indices):
                    # Get pixel center in geographic coordinates
                    lon, lat = transform_1km * (col + 0.5, row + 0.5)
                    point = Point(lon, lat)
                    
                    # Create buffer (approximate for small areas: ~111km per degree at equator)
                    # For Nigeria (latitude ~6-13°N), 1 degree ≈ 111km
                    buffer_degrees = buffer_radius_km / 111.0
                    buffer_poly = point.buffer(buffer_degrees)
                    polygons.append(buffer_poly)
                
                # Merge overlapping buffers
                merged_polygon = unary_union(polygons)
                
                # Convert to GeoDataFrame
                if isinstance(merged_polygon, Polygon):
                    polygons_list = [merged_polygon]
                elif isinstance(merged_polygon, MultiPolygon):
                    polygons_list = list(merged_polygon.geoms)
                else:
                    polygons_list = []
                
                gdf_buffer = gpd.GeoDataFrame(
                    {
                        'city': [city] * len(polygons_list),
                        'method': [method] * len(polygons_list),
                        'period': [period] * len(polygons_list),
                        'n_pixels': [n_pixels_original] * len(polygons_list),
                        'delineation_path': ['BUFFER'] * len(polygons_list)
                    },
                    geometry=polygons_list,
                    crs=crs
                )
                
                # Store buffer polygons
                isolated_hotspot_polygons[city][key] = gdf_buffer
                
                # Track statistics
                buffer_stats.append({
                    'City': city,
                    'Method': method.upper(),
                    'Period': period.capitalize(),
                    'Original_Pixels': n_pixels_original,
                    'Buffer_Polygons': len(polygons_list),
                    'Path': 'BUFFER'
                })
                
                print(f"  {method.upper():10s} {period.capitalize():10s}: BUFFER path → {n_pixels_original:4d} pixels → {len(polygons_list)} polygon(s)")

print("\n" + "="*70)
print("DELINEATION SUMMARY")
print("="*70)

# KDE Path Summary
if resampling_stats:
    resampling_df = pd.DataFrame(resampling_stats)
    print(f"\n📊 KDE PATH (≥{min_pixels} pixels):")
    print(f"  Total cases: {len(resampling_df)}")
    print(f"  Total original pixels: {resampling_df['Original_Pixels'].sum():,}")
    print(f"  Total resampled pixels: {resampling_df['Resampled_Pixels'].sum():,}")
else:
    print(f"\n📊 KDE PATH: No hotspots met threshold (≥{min_pixels} pixels)")

# Buffer Path Summary
if buffer_stats:
    buffer_df = pd.DataFrame(buffer_stats)
    print(f"\n🎯 BUFFER PATH (<{min_pixels} pixels):")
    print(f"  Total cases: {len(buffer_df)}")
    print(f"  Total original pixels: {buffer_df['Original_Pixels'].sum():,}")
    print(f"  Total buffer polygons: {buffer_df['Buffer_Polygons'].sum():,}")
else:
    print(f"\n🎯 BUFFER PATH: No isolated hotspots (<{min_pixels} pixels)")

print("\n" + "="*70)
print("✅ Two-tier delineation complete")
print(f"  • {len(resampling_stats)} cases → KDE path (Section 3.2)")
print(f"  • {len(buffer_stats)} cases → Direct buffer polygons")
print("="*70 + "\n")

### 3.2 KDE Bandwidth Calculation and Density Surfaces

Apply Scott's rule bandwidth selection, fit the KDE model on resampled coordinates, and evaluate on an adaptive grid to produce smooth density surfaces.

In [ ]:
print("="*70)
print("🧮 SECTION 3.2: KDE BANDWIDTH & DENSITY SURFACE GENERATION")
print("="*70)

# Get KDE configuration
bandwidth_method = KDE_CONFIG['bandwidth_method']
grid_resolution_factor = KDE_CONFIG['grid_resolution_factor']
max_grid_size = KDE_CONFIG['max_grid_size']

print(f"\nKDE Configuration:")
print(f"  Bandwidth method: {bandwidth_method}")
print(f"  Grid resolution factor: {grid_resolution_factor}x")
print(f"  Max grid size: {max_grid_size}×{max_grid_size}")

# ============================================================================
# CACHE SETUP: Skip expensive KDE computation if already done
# ============================================================================
import hashlib

# Define cache directory
kde_cache_dir = OUTPUT_DIR / '05_hotspot_analysis' / 'kde_cache'
kde_cache_dir.mkdir(parents=True, exist_ok=True)

# Cache manifest file
cache_manifest_path = kde_cache_dir / 'kde_cache_manifest.json'

# Load existing cache manifest
if cache_manifest_path.exists():
    with open(cache_manifest_path, 'r') as f:
        cache_manifest = json.load(f)
    print(f"\n📁 Loaded cache manifest: {len(cache_manifest)} entries")
else:
    cache_manifest = {}
    print(f"\n📁 No existing cache found, will create new cache")

def compute_data_hash(coords_array):
    """Compute hash of input data and configuration to detect changes"""
    # Hash coordinates + relevant config parameters
    data_str = (
        coords_array.tobytes() +
        f"{RESAMPLE_CONFIG['resolution_m']}".encode() +
        f"{KDE_CONFIG['bandwidth_method']}".encode() +
        f"{KDE_CONFIG['grid_resolution_factor']}".encode()
    )
    return hashlib.md5(data_str).hexdigest()

# Storage for KDE results
kde_surfaces = {}
kde_stats = []
cache_hits = 0
cache_misses = 0

# Process each city's hotspots
for city in tqdm(CITIES, desc="KDE Fitting", unit="city"):
    if city not in hotspot_coordinates or len(hotspot_coordinates[city]) == 0:
        print(f"\n🏙️  {city}: No hotspots to process")
        continue
    
    print(f"\n🏙️  {city}")
    print("-" * 60)
    
    kde_surfaces[city] = {}
    
    # Use nested progress bar for methods within each city
    for key, data in tqdm(list(hotspot_coordinates[city].items()), desc=f"  {city} Methods", leave=False, unit="method"):
        coords = data['coordinates']
        method = data['method']
        period = data['period']
        n_pixels = len(coords)
        
        print(f"\n  {method.upper():10s} {period.capitalize():10s}")
        
        # ====================================================================
        # CHECK CACHE: Skip if already computed with same data
        # ====================================================================
        cache_key = f"{city}_{method}_{period}"
        data_hash = compute_data_hash(coords)
        
        # Check if cached and hash matches
        if cache_key in cache_manifest:
            cached_entry = cache_manifest[cache_key]
            if cached_entry.get('data_hash') == data_hash:
                # Load from cache
                cache_file = kde_cache_dir / f"{cache_key}_kde.npz"
                if cache_file.exists():
                    try:
                        cached_data = np.load(cache_file, allow_pickle=True)
                        
                        # Extract cached values for display
                        bandwidth = float(cached_data['bandwidth'])
                        n_lon, n_lat = cached_data['grid_size']
                        density_normalized = cached_data['density_normalized']
                        density_min = density_normalized.min() * (density_normalized.max() - density_normalized.min())
                        density_max = density_normalized.max() * (density_normalized.max() - density_normalized.min())
                        
                        # Display same information as fresh computation
                        print(f"    📦 Loaded from cache (hash: {data_hash[:8]}...)")
                        print(f"    ✓ KDE fitted: {n_pixels:,} points")
                        print(f"    ✓ Bandwidth (Scott): {bandwidth:.6f}°")
                        print(f"    ✓ Grid size: {n_lon}×{n_lat} = {n_lon*n_lat:,} points")
                        print(f"    ✓ Density normalized to [0, 1]")
                        
                        # Restore KDE surface from cache
                        kde_surfaces[city][key] = {
                            'kde_model': None,  # Don't store full model (large)
                            'bandwidth': bandwidth,
                            'lon_grid': cached_data['lon_grid'],
                            'lat_grid': cached_data['lat_grid'],
                            'lon_mesh': cached_data['lon_mesh'],
                            'lat_mesh': cached_data['lat_mesh'],
                            'density_normalized': density_normalized,
                            'extent': tuple(cached_data['extent']),
                            'grid_size': tuple(cached_data['grid_size']),
                            'method': method,
                            'period': period,
                            'n_pixels': n_pixels
                        }
                        
                        # Track statistics
                        kde_stats.append({
                            'City': city,
                            'Method': method.upper(),
                            'Period': period.capitalize(),
                            'N_Points': n_pixels,
                            'Bandwidth': bandwidth,
                            'Grid_Size': f"{n_lon}×{n_lat}",
                            'Total_Grid_Points': n_lon * n_lat,
                            'Status': '📦 Cached'
                        })
                        
                        cache_hits += 1
                        continue
                        
                    except Exception as e:
                        print(f"    ⚠️  Cache load failed: {e}, recomputing...")
        
        # ====================================================================
        # COMPUTE KDE: Cache miss or invalid cache
        # ====================================================================
        cache_misses += 1
        
        # Transpose for KDE (needs shape: (2, n_samples))
        coords_transposed = coords.T
        
        try:
            # Fit KDE model using Scott's rule
            kde = gaussian_kde(coords_transposed, bw_method=bandwidth_method)
            
            # Get bandwidth value (covariance factor)
            bandwidth = kde.covariance_factor()
            
            print(f"    ✓ KDE fitted: {n_pixels:,} points")
            print(f"    ✓ Bandwidth (Scott): {bandwidth:.6f}°")
            
            # Define adaptive grid based on coordinate extent
            lon_min, lat_min = coords.min(axis=0)
            lon_max, lat_max = coords.max(axis=0)
            
            # Add 10% buffer to extent
            lon_buffer = (lon_max - lon_min) * 0.10
            lat_buffer = (lat_max - lat_min) * 0.10
            
            lon_extent = [lon_min - lon_buffer, lon_max + lon_buffer]
            lat_extent = [lat_min - lat_buffer, lat_max + lat_buffer]
            
            # Calculate grid size based on resolution factor
            # Grid spacing = resampled pixel size / resolution factor
            grid_spacing_deg = (resample_resolution_m / 111000) / grid_resolution_factor  # ~111km per degree
            
            n_lon = int((lon_extent[1] - lon_extent[0]) / grid_spacing_deg)
            n_lat = int((lat_extent[1] - lat_extent[0]) / grid_spacing_deg)
            
            # Limit grid size to prevent memory issues
            n_lon = min(n_lon, max_grid_size)
            n_lat = min(n_lat, max_grid_size)
            
            print(f"    ✓ Grid size: {n_lon}×{n_lat} = {n_lon*n_lat:,} points")
            
            # Create meshgrid
            lon_grid = np.linspace(lon_extent[0], lon_extent[1], n_lon)
            lat_grid = np.linspace(lat_extent[0], lat_extent[1], n_lat)
            lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
            
            # Evaluate KDE on grid
            grid_coords = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])
            density = kde(grid_coords).reshape(lon_mesh.shape)
            
            # Normalize density to [0, 1]
            density_normalized = (density - density.min()) / (density.max() - density.min() + 1e-10)
            
            print(f"    ✓ Density normalized to [0, 1]")
            
            # Store KDE surface
            kde_surfaces[city][key] = {
                'kde_model': None,  # Don't store full model (large)
                'bandwidth': bandwidth,
                'lon_grid': lon_grid,
                'lat_grid': lat_grid,
                'lon_mesh': lon_mesh,
                'lat_mesh': lat_mesh,
                'density_normalized': density_normalized,
                'extent': (lon_extent[0], lon_extent[1], lat_extent[0], lat_extent[1]),
                'grid_size': (n_lon, n_lat),
                'method': method,
                'period': period,
                'n_pixels': n_pixels
            }
            
            # Track statistics
            kde_stats.append({
                'City': city,
                'Method': method.upper(),
                'Period': period.capitalize(),
                'N_Points': n_pixels,
                'Bandwidth': bandwidth,
                'Grid_Size': f"{n_lon}×{n_lat}",
                'Total_Grid_Points': n_lon * n_lat,
                'Status': '🆕 New'
            })
            
            # ====================================================================
            # SAVE TO CACHE
            # ====================================================================
            cache_file = kde_cache_dir / f"{cache_key}_kde.npz"
            np.savez_compressed(
                cache_file,
                lon_grid=lon_grid,
                lat_grid=lat_grid,
                lon_mesh=lon_mesh,
                lat_mesh=lat_mesh,
                density_normalized=density_normalized,
                extent=np.array([lon_extent[0], lon_extent[1], lat_extent[0], lat_extent[1]]),
                grid_size=np.array([n_lon, n_lat]),
                bandwidth=bandwidth
            )
            
            # Update manifest
            cache_manifest[cache_key] = {
                'data_hash': data_hash,
                'timestamp': datetime.now().isoformat(),
                'n_pixels': n_pixels,
                'bandwidth': bandwidth,
                'grid_size': [n_lon, n_lat]
            }
            
        except Exception as e:
            print(f"    ✗ KDE computation failed: {str(e)}")
            continue

# Save updated cache manifest
with open(cache_manifest_path, 'w') as f:
    json.dump(cache_manifest, f, indent=2)

print("\n" + "="*70)
print("KDE COMPUTATION SUMMARY")
print("="*70)

print(f"\nCache Performance:")
print(f"  Cache hits: {cache_hits}")
print(f"  Cache misses: {cache_misses}")
print(f"  Cache hit rate: {cache_hits/(cache_hits+cache_misses)*100:.1f}%" if (cache_hits+cache_misses) > 0 else "  N/A")

if len(kde_stats) > 0:
    kde_stats_df = pd.DataFrame(kde_stats)
    
    print(f"\nKDE Surfaces Generated: {len(kde_stats_df)}")
    print(f"Total grid points: {kde_stats_df['Total_Grid_Points'].sum():,}")
    
    print("\nSample of KDE statistics:")
    print(kde_stats_df.head(10).to_string(index=False))

print("\n" + "="*70)
print("✅ KDE bandwidth calculation & density surface generation complete")
print("="*70 + "\n")

### 3.3 Extract Contours and Merge with Buffer Polygons

Extract nested density contours at 50 %, 75 %, and 90 % from KDE surfaces and combine them with the pre-generated buffer polygons from Section 3.1 to produce a unified polygon dataset.

In [ ]:
print("="*70)
print("🗺️  SECTION 3.3: EXTRACT CONTOURS & MERGE POLYGONS")
print("="*70)

# Get contour configuration
contour_levels = KDE_CONFIG['contour_levels']
simplify_tolerance = KDE_CONFIG['simplify_tolerance']
min_polygon_area_km2 = KDE_CONFIG['min_polygon_area_km2']
resample_resolution_m = RESAMPLE_CONFIG['resolution_m']  # Needed for feature metadata

print(f"\n{'='*60}")
print("TWO-PATH POLYGON GENERATION")
print(f"{'='*60}")
print(f"\n📊 KDE PATH (from Section 3.2):")
print(f"  Contour levels: {contour_levels}")
print(f"  Simplify tolerance: {simplify_tolerance}° (~{simplify_tolerance*111:.1f}m)")
print(f"  Min polygon area: {min_polygon_area_km2} km²")
print(f"  Total KDE surfaces: {sum(len(kde_surfaces[c]) for c in kde_surfaces)}")

print(f"\n🎯 BUFFER PATH (from Section 3.1):")
print(f"  Total buffer polygon sets: {sum(len(isolated_hotspot_polygons[c]) for c in isolated_hotspot_polygons)}")
print(f"  Source: Pre-generated 500m buffers (isolated hotspots)")

print(f"\n  → Both paths will be merged into unified polygon dataset")
print(f"{'='*60}\n")

# Storage for polygon features
polygon_features = []
contour_stats = []

# Process each KDE surface with progress tracking
for city in tqdm(kde_surfaces.keys(), desc="Extracting Contours", unit="city"):
    print(f"\n🏙️  {city}")
    print("-" * 60)
    
    # Use nested progress bar for methods
    for key, kde_data in tqdm(list(kde_surfaces[city].items()), desc=f"  {city} Methods", leave=False, unit="method"):
        method = kde_data['method']
        period = kde_data['period']
        density = kde_data['density_normalized']
        lon_grid = kde_data['lon_grid']
        lat_grid = kde_data['lat_grid']
        
        print(f"\n  {method.upper():10s} {period.capitalize():10s}")
        
        # Extract contours at each level
        for contour_level in contour_levels:
            try:
                # Find contours at this level
                contours = measure.find_contours(density, level=contour_level)
                
                if len(contours) == 0:
                    print(f"    {contour_level:4.0%} level: No contours found")
                    continue
                
                print(f"    {contour_level:4.0%} level: {len(contours)} contour(s) found")
                
                # Convert each contour to polygon
                valid_polygons = []
                for contour_idx, contour in enumerate(contours):
                    # Convert pixel coordinates to geographic coordinates
                    # contour format: (row, col) indices
                    rows = contour[:, 0]
                    cols = contour[:, 1]
                    
                    # Interpolate to get lat/lon
                    lons = np.interp(cols, np.arange(len(lon_grid)), lon_grid)
                    lats = np.interp(rows, np.arange(len(lat_grid)), lat_grid)
                    
                    # Create coordinate pairs
                    coords = list(zip(lons, lats))
                    
                    # Need at least 3 points for polygon
                    if len(coords) < 3:
                        continue
                    
                    # Create polygon
                    try:
                        poly = Polygon(coords)
                        
                        # Validate and fix geometry
                        if not poly.is_valid:
                            poly = poly.buffer(0)  # Fix self-intersections
                        
                        # Simplify polygon to reduce vertices
                        if simplify_tolerance > 0:
                            poly = poly.simplify(simplify_tolerance, preserve_topology=True)
                        
                        # Check minimum area
                        area_km2 = poly.area * (111 * 111)  # Rough conversion deg² to km²
                        
                        if area_km2 < min_polygon_area_km2:
                            continue
                        
                        valid_polygons.append((poly, area_km2))
                        
                    except Exception as e:
                        continue
                
                if len(valid_polygons) == 0:
                    print(f"              → 0 valid polygons (area filter)")
                    continue
                
                print(f"              → {len(valid_polygons)} valid polygon(s)")
                
                # Store features
                for poly_idx, (poly, area_km2) in enumerate(valid_polygons):
                    # Generate unique ID
                    hotspot_id = f"{city}_{method}_{period}_{int(contour_level*100)}_p{poly_idx+1}"
                    
                    # Get original pixel counts
                    n_pixels_original = hotspot_coordinates[city][key]['n_pixels_original']
                    n_pixels_resampled = hotspot_coordinates[city][key]['n_pixels_resampled']
                    bandwidth = kde_data['bandwidth']
                    
                    feature = {
                        'hotspot_id': hotspot_id,
                        'city': city,
                        'method': method.upper(),
                        'period': period.capitalize(),
                        'contour_level': contour_level,
                        'polygon_index': poly_idx + 1,
                        'n_pixels_original': n_pixels_original,
                        'n_pixels_resampled': n_pixels_resampled,
                        'kde_bandwidth': bandwidth,
                        'resample_resolution_m': resample_resolution_m,
                        'area_km2': area_km2,
                        'perimeter_km': poly.length * 111,  # Rough conversion
                        'n_vertices': len(poly.exterior.coords),
                        'delineation_method': 'kde_contour',
                        'geometry': poly
                    }
                    
                    polygon_features.append(feature)
                    
                    # Track statistics
                    contour_stats.append({
                        'City': city,
                        'Method': method.upper(),
                        'Period': period.capitalize(),
                        'Contour_Level': f"{contour_level:.0%}",
                        'N_Polygons': 1,
                        'Area_km2': area_km2
                    })
                    
            except Exception as e:
                print(f"    {contour_level:4.0%} level: ✗ Failed - {str(e)}")
                continue

print("\n" + "="*70)
print("CONTOUR EXTRACTION SUMMARY")
print("="*70)

print(f"\nTotal polygon features created: {len(polygon_features)}")

if len(polygon_features) > 0:
    contour_stats_df = pd.DataFrame(contour_stats)
    
    # Aggregate statistics
    print("\nPolygons by contour level:")
    level_summary = contour_stats_df.groupby('Contour_Level').agg({
        'N_Polygons': 'sum',
        'Area_km2': ['sum', 'mean']
    }).round(2)
    print(level_summary)
    
    print("\nPolygons by method:")
    method_summary = contour_stats_df.groupby('Method').agg({
        'N_Polygons': 'sum',
        'Area_km2': ['sum', 'mean']
    }).round(2)
    print(method_summary)

# ========================================================================
# MERGE BUFFER POLYGONS (Section 3.1 isolated hotspots)
# ========================================================================
print("\n" + "="*70)
print("MERGING BUFFER POLYGONS")
print("="*70)

buffer_polygon_count = 0
for city in isolated_hotspot_polygons.keys():
    if len(isolated_hotspot_polygons[city]) == 0:
        continue
    
    print(f"\n🏙️  {city}")
    
    for key, gdf_buffer in isolated_hotspot_polygons[city].items():
        method = gdf_buffer['method'].iloc[0]
        period = gdf_buffer['period'].iloc[0]
        n_polygons = len(gdf_buffer)
        
        print(f"  {method.upper():10s} {period.capitalize():10s}: {n_polygons} buffer polygon(s)")
        
        # Add buffer polygons to main feature list
        for idx, row in gdf_buffer.iterrows():
            hotspot_id = f"{city}_{method}_{period}_buffer_{idx+1}"
            poly = row.geometry
            area_km2 = poly.area * (111 ** 2)  # Rough conversion
            
            feature = {
                'hotspot_id': hotspot_id,
                'city': city,
                'method': method.upper(),
                'period': period.capitalize(),
                'contour_level': 1.0,  # Buffer = 100% coverage
                'polygon_index': idx + 1,
                'n_pixels_original': row['n_pixels'],
                'n_pixels_resampled': 0,  # Not applicable
                'kde_bandwidth': 0,  # Not applicable
                'resample_resolution_m': resample_resolution_m,
                'area_km2': area_km2,
                'perimeter_km': poly.length * 111,
                'n_vertices': len(poly.exterior.coords) if isinstance(poly, Polygon) else 0,
                'delineation_method': 'buffer_500m',
                'geometry': poly
            }
            
            polygon_features.append(feature)
            buffer_polygon_count += 1

print(f"\n✓ Added {buffer_polygon_count} buffer polygons")
print(f"✓ Total polygons: {len(polygon_features)} (KDE + Buffer)")

print("\n" + "="*70)
print("✅ Contour extraction & buffer merging complete")
print("="*70 + "\n")

### 3.4 Build Unified GeoDataFrame and Clean Memory

Compile all hotspot polygons (KDE contours and buffer zones) into a single GeoDataFrame with attributes for delineation method, detection method, period, area, perimeter, and source metadata. Temporary resampled arrays and KDE surfaces are deleted to free memory.

In [ ]:
print("="*70)
print("📦 SECTION 3.4: BUILD GEODATAFRAME & CLEAN MEMORY")
print("="*70)

# Create GeoDataFrame from polygon features
if len(polygon_features) > 0:
    hotspot_polygons_gdf = gpd.GeoDataFrame(
        polygon_features,
        crs=KDE_CONFIG['output_crs'],
        geometry='geometry'
    )
    
    print(f"\n✓ GeoDataFrame created: {len(hotspot_polygons_gdf)} features")
    print(f"  CRS: {hotspot_polygons_gdf.crs}")
    print(f"  Columns: {len(hotspot_polygons_gdf.columns)}")
    
    # Display schema
    print("\nGeoDataFrame Schema:")
    print("-" * 60)
    for col in hotspot_polygons_gdf.columns:
        dtype = hotspot_polygons_gdf[col].dtype
        print(f"  {col:30s} : {dtype}")
    
    # Summary statistics
    print("\n" + "="*60)
    print("HOTSPOT POLYGON SUMMARY")
    print("="*60)
    
    print(f"\nTotal Features: {len(hotspot_polygons_gdf)}")
    print(f"Total Area: {hotspot_polygons_gdf['area_km2'].sum():.2f} km²")
    print(f"Mean Area: {hotspot_polygons_gdf['area_km2'].mean():.2f} km²")
    print(f"Min Area: {hotspot_polygons_gdf['area_km2'].min():.2f} km²")
    print(f"Max Area: {hotspot_polygons_gdf['area_km2'].max():.2f} km²")
    
    # Breakdown by city
    print("\nFeatures by City:")
    city_counts = hotspot_polygons_gdf.groupby('city').agg({
        'hotspot_id': 'count',
        'area_km2': 'sum'
    }).rename(columns={'hotspot_id': 'N_Features', 'area_km2': 'Total_Area_km2'})
    print(city_counts.to_string())
    
    # Breakdown by method
    print("\nFeatures by Detection Method:")
    method_counts = hotspot_polygons_gdf.groupby('method').agg({
        'hotspot_id': 'count',
        'area_km2': 'sum'
    }).rename(columns={'hotspot_id': 'N_Features', 'area_km2': 'Total_Area_km2'})
    print(method_counts.to_string())
    
    # Breakdown by contour level
    print("\nFeatures by Contour Level:")
    contour_counts = hotspot_polygons_gdf.groupby('contour_level').agg({
        'hotspot_id': 'count',
        'area_km2': ['sum', 'mean']
    }).round(2)
    print(contour_counts.to_string())
    
    # Display sample features
    print("\nSample Features (first 5):")
    print("="*100)
    sample_cols = ['hotspot_id', 'city', 'method', 'period', 'contour_level', 
                   'n_pixels_original', 'area_km2', 'n_vertices']
    print(hotspot_polygons_gdf[sample_cols].head().to_string(index=False))
    
else:
    print("\n⚠️  No polygon features were created")
    hotspot_polygons_gdf = gpd.GeoDataFrame(columns=['geometry'], crs=KDE_CONFIG['output_crs'])

# Free memory: Delete temporary data structures
print("\n" + "="*70)
print("MEMORY CLEANUP")
print("="*70)

# Delete resampled coordinate arrays
del hotspot_coordinates
print("✓ Deleted: hotspot_coordinates (resampled pixel arrays)")

# Delete KDE surfaces (large grid arrays)
del kde_surfaces
print("✓ Deleted: kde_surfaces (KDE density grids)")

# Delete intermediate DataFrames
del resampling_df, kde_stats_df, contour_stats_df
print("✓ Deleted: Intermediate statistics DataFrames")

# Force garbage collection
import gc
gc.collect()
print("✓ Garbage collection completed")

print("\n" + "="*70)
print("✅ Section 3: KDE Delineation Complete")
print("="*70)
print(f"\nFinal Output:")
print(f"  Variable: hotspot_polygons_gdf")
print(f"  Type: GeoDataFrame")
print(f"  Features: {len(hotspot_polygons_gdf)}")
print(f"  CRS: {hotspot_polygons_gdf.crs if len(hotspot_polygons_gdf) > 0 else 'N/A'}")
print(f"  Memory: Resampled arrays freed, only polygons retained")
print("="*70 + "\n")

### 3.5 Visualisation: KDE Contour Polygons

Quick-look visualisation of P90 Day hotspot polygons overlaid on LST rasters for all cities to verify delineation quality.

In [ ]:
# Quick visualization of KDE polygons (sample: P90 Day for all cities)
if len(hotspot_polygons_gdf) > 0:
    print("="*70)
    print("🎨 KDE POLYGON VISUALIZATION (Sample: P90 Day)")
    print("="*70 + "\n")
    
    # Filter for P90 Day hotspots
    sample_gdf = hotspot_polygons_gdf[
        (hotspot_polygons_gdf['method'] == 'P90') & 
        (hotspot_polygons_gdf['period'] == 'Day')
    ].copy()
    
    if len(sample_gdf) > 0:
        fig, axes = plt.subplots(1, len(CITIES), figsize=(4*len(CITIES), 4))
        if len(CITIES) == 1:
            axes = [axes]
        
        for idx, city in enumerate(CITIES):
            ax = axes[idx]
            
            # Get city polygons
            city_polys = sample_gdf[sample_gdf['city'] == city]
            
            if len(city_polys) == 0:
                ax.text(0.5, 0.5, f'{city}\nNo P90 Day\npolygons', 
                       ha='center', va='center', transform=ax.transAxes)
                ax.set_title(city)
                ax.axis('off')
                continue
            
            # Get LST raster for background
            day_data = persistent_heat_rasters[city]['day']['data']
            day_transform = persistent_heat_rasters[city]['day']['transform']
            day_display = day_data.copy().astype(float)
            day_display[day_data <= 0] = np.nan
            
            # Calculate extent from transform for proper georeferencing
            # extent = [left, right, bottom, top] in geographic coordinates
            from rasterio.transform import array_bounds
            bounds = array_bounds(day_data.shape[0], day_data.shape[1], day_transform)
            extent = [bounds[0], bounds[2], bounds[1], bounds[3]]  # [xmin, xmax, ymin, ymax]
            
            # Plot LST with proper extent (now in geographic coordinates)
            im = ax.imshow(day_display, cmap='RdYlBu_r', alpha=0.5, 
                          extent=extent, aspect='auto', origin='upper')
            
            # Plot polygons by contour level (now coordinates will align!)
            colors = {0.50: 'yellow', 0.75: 'orange', 0.90: 'red'}
            for level in [0.50, 0.75, 0.90]:
                level_polys = city_polys[city_polys['contour_level'] == level]
                if len(level_polys) > 0:
                    level_polys.plot(ax=ax, facecolor='none', edgecolor=colors[level], 
                                    linewidth=2, label=f'{level:.0%}')
            
            ax.set_title(f'{city} - P90 Day\n{len(city_polys)} polygons', fontsize=10)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')
            if idx == 0:
                ax.legend(loc='upper left', fontsize=8, title='Contour')
        
        plt.tight_layout()
        plt.show()
        
        print(f"✓ Displayed KDE polygons for {len(CITIES)} cities")
    else:
        print("⚠️  No P90 Day polygons to visualize")
else:
    print("⚠️  No polygons to visualize")

print("\n")

---

## Section 4: Attribute Enrichment and Export

Enrich each KDE-delineated hotspot polygon with landscape character and thermal attributes using modularised functions from `src/data_io.py`.

**Data integration pipeline:**

1. **Landscape character** (10 columns) -- area-weighted dominant class and coverage percentage for Elevation, Climate, Landcover, Geology, and Cluster25, extracted via spatial overlay with the Nigeria-wide physiographic classification.
2. **LST statistics** (12 columns) -- min, mean, max, std, median, and pixel count for both day and night rasters, computed via zonal statistics.
3. **Multi-CRS export** -- GeoJSON files in EPSG:4326 (WGS84) and EPSG:3857 (Web Mercator).

### 4.1 Load Landscape Data

Load the physiographic landscape GeoJSON via `data_io.load_landscape_data()` and reproject to EPSG:4326 for spatial overlay.

In [ ]:
print("="*70)
print("📂 SECTION 4: COMPREHENSIVE GEODATAFRAME DEVELOPMENT")
print("="*70 + "\n")

print("Loading data sources...")
print("-" * 70)

# Import modular function
from src.data_io import load_landscape_data

# Load landscape GeoJSON using modular function (automatically handles path and reprojection)
landscape_gdf_4326 = load_landscape_data(target_crs='EPSG:4326')

print(f"✓ Landscape data loaded via data_io.load_landscape_data()")
print(f"  - Features: {len(landscape_gdf_4326):,}")
print(f"  - CRS: {landscape_gdf_4326.crs}")

# Get class attributes from config (already imported in first cell)
class_attributes = LANDSCAPE_CLASS_ATTRIBUTES

print(f"  - Class attributes (from config): {', '.join(class_attributes)}")
print(f"  - Cluster attribute (from config): {LANDSCAPE_CLUSTER_ATTRIBUTE}")

# Verify we have polygons to process
print(f"\n✓ Hotspot polygons ready: {len(hotspot_polygons_gdf):,} features")
print(f"  - Cities: {', '.join(sorted(hotspot_polygons_gdf['city'].unique()))}")
print(f"  - Methods: {', '.join(sorted(hotspot_polygons_gdf['method'].unique()))}")
print(f"  - Periods: {', '.join(sorted(hotspot_polygons_gdf['period'].unique()))}")

### 4.2 Landscape Character Integration

Extract dominant landscape class and coverage percentage for each polygon via `data_io.extract_landscape_attributes()`. Attributes include Elevation, Climate, Landcover, Geology, and the 25-class physiographic cluster.

In [ ]:
# Import modular function
from src.data_io import extract_landscape_attributes

# Process all polygons
print("🌍 Extracting landscape character attributes...")
print("-" * 70)

landscape_attributes_list = []

for idx, row in tqdm(hotspot_polygons_gdf.iterrows(), total=len(hotspot_polygons_gdf), 
                      desc="Polygons", unit="poly"):
    # Create single-feature GeoDataFrame
    poly_gdf = gpd.GeoDataFrame([row], crs=hotspot_polygons_gdf.crs)
    
    # Extract landscape attributes using modular function
    attrs = extract_landscape_attributes(poly_gdf, landscape_gdf_4326, class_attributes)
    landscape_attributes_list.append(attrs)

# Convert to DataFrame
landscape_df = pd.DataFrame(landscape_attributes_list)

print(f"\n✓ Landscape attributes extracted for {len(landscape_df):,} polygons")
print(f"  - New columns: {len(landscape_df.columns)}")
print(f"  - Sample attributes: {list(landscape_df.columns[:5])}")

# Preview
print(f"\n📊 Sample landscape attributes:")
print(landscape_df.head(3).to_string())

print("\n" + "="*70 + "\n")

### 4.3 LST Statistics Extraction

Compute zonal LST statistics (min, mean, max, std, median, pixel count) for day and night rasters within each hotspot polygon via `data_io.extract_lst_statistics()`.

In [ ]:
# Import modular functions
from src.data_io import get_lst_raster_files, extract_lst_statistics
from config import LST_RASTER_CONFIG

# Process all polygons
print("🌡️  Extracting LST statistics...")
print("-" * 70)

lst_stats_list = []

# Group polygons by city for efficient raster loading
city_groups = hotspot_polygons_gdf.groupby('city')

for city, city_polys in tqdm(city_groups, desc="Cities", unit="city"):
    # Get LST raster files for this city using modular function
    day_raster, night_raster = get_lst_raster_files(city, seviri_base_path)
    
    if day_raster is None or night_raster is None:
        print(f"  ⚠️ LST rasters not found for {city}")
        # Add null stats
        for _ in range(len(city_polys)):
            lst_stats_list.append({
                'LST_day_min': None, 'LST_day_mean': None, 'LST_day_max': None,
                'LST_day_std': None, 'LST_day_median': None, 'LST_day_n_pixels': 0,
                'LST_night_min': None, 'LST_night_mean': None, 'LST_night_max': None,
                'LST_night_std': None, 'LST_night_median': None, 'LST_night_n_pixels': 0
            })
        continue
    
    # Process each polygon in this city
    for idx, row in city_polys.iterrows():
        poly_gdf = gpd.GeoDataFrame([row], crs=hotspot_polygons_gdf.crs)
        
        # Extract LST stats using modular function (with config-driven valid range)
        day_stats = extract_lst_statistics(
            poly_gdf, day_raster, 
            valid_range=LST_RASTER_CONFIG['valid_range']
        )
        night_stats = extract_lst_statistics(
            poly_gdf, night_raster,
            valid_range=LST_RASTER_CONFIG['valid_range']
        )
        
        # Combine stats
        combined_stats = {}
        if day_stats:
            combined_stats.update({f'LST_day_{k}': v for k, v in day_stats.items()})
        else:
            combined_stats.update({
                'LST_day_min': None, 'LST_day_mean': None, 'LST_day_max': None,
                'LST_day_std': None, 'LST_day_median': None, 'LST_day_n_pixels': 0
            })
        
        if night_stats:
            combined_stats.update({f'LST_night_{k}': v for k, v in night_stats.items()})
        else:
            combined_stats.update({
                'LST_night_min': None, 'LST_night_mean': None, 'LST_night_max': None,
                'LST_night_std': None, 'LST_night_median': None, 'LST_night_n_pixels': 0
            })
        
        lst_stats_list.append(combined_stats)

# Convert to DataFrame
lst_stats_df = pd.DataFrame(lst_stats_list)

print(f"\n✓ LST statistics extracted for {len(lst_stats_df):,} polygons")
print(f"  - New columns: {len(lst_stats_df.columns)}")
print(f"  - Day stats: min, mean, max, std, median, n_pixels")
print(f"  - Night stats: min, mean, max, std, median, n_pixels")

# Preview
print(f"\n📊 Sample LST statistics (°C):")
sample_cols = ['LST_day_mean', 'LST_day_std', 'LST_night_mean', 'LST_night_std', 'LST_day_n_pixels']
print(lst_stats_df[sample_cols].head(3).to_string())

print("\n" + "="*70 + "\n")

### 4.4 Population Enrichment (GHSL)

Estimate resident population within each hotspot polygon using the JRC Global Human Settlement Layer (GHSL R2023A, `GHS_POP`). The national Nigeria raster is downloaded once from Google Earth Engine at 1 km resolution (100 m aggregated via sum) and stored permanently in `data/inputs/geospatial_features/`. Zonal sums are computed per polygon via `data_io.extract_population_statistics()`.

New columns: `pop_total`, `pop_density_km2`, `ghsl_year`.

In [ ]:
import importlib
import src.data_io
importlib.reload(src.data_io)
from src.data_io import download_ghsl_population_raster, extract_population_statistics

print("="*70)
print("SECTION 4.4: GHSL POPULATION ENRICHMENT")
print("="*70 + "\n")

# --- Skip logic: check if population columns already exist ---
if 'pop_total' in hotspot_polygons_gdf.columns and hotspot_polygons_gdf['pop_total'].notna().all():
    print("✓ Population columns already present in hotspot_polygons_gdf — skipping extraction")
    pop_stats_df = hotspot_polygons_gdf[['pop_total', 'pop_density_km2']].copy()
    print(f"  - pop_total  range: {pop_stats_df['pop_total'].min():.0f} – {pop_stats_df['pop_total'].max():.0f}")
    print(f"  - pop_density range: {pop_stats_df['pop_density_km2'].min():.0f} – {pop_stats_df['pop_density_km2'].max():.0f} people/km²")
else:
    # Step 1: Download national GHSL raster (skip if file already exists)
    print("[1/2] Downloading GHSL population raster ...")
    print("-" * 70)

    ghsl_raster_path = download_ghsl_population_raster(
        ghsl_year=ghsl_year,
        ghsl_pop_config=GHSL_POP_CONFIG,
        gee_config=GEE_CONFIG,
    )

    # Step 2: Extract population per polygon
    print(f"\n[2/2] Extracting population for {len(hotspot_polygons_gdf):,} polygons ...")
    print("-" * 70)

    pop_stats_df = extract_population_statistics(
        polygon_gdf=hotspot_polygons_gdf,
        ghsl_raster_path=ghsl_raster_path,
    )

    # Summary
    valid_count = pop_stats_df['pop_total'].notna().sum()
    print(f"\n✓ Population extracted for {valid_count:,}/{len(pop_stats_df):,} polygons")
    print(f"  - pop_total  range: {pop_stats_df['pop_total'].min():.0f} – {pop_stats_df['pop_total'].max():.0f}")
    print(f"  - pop_density range: {pop_stats_df['pop_density_km2'].min():.0f} – {pop_stats_df['pop_density_km2'].max():.0f} people/km²")

    # Quick city-level overview
    temp_df = hotspot_polygons_gdf[['city']].copy()
    temp_df['pop_total'] = pop_stats_df['pop_total'].values
    city_pop = temp_df.groupby('city')['pop_total'].sum()
    print(f"\n  Population by city:")
    for city_name, total in city_pop.items():
        print(f"    {city_name}: {total:,.0f}")

print("\n" + "="*70 + "\n")

### 4.5 Combine Attributes and Export

Merge landscape, LST, and population columns into the polygon GeoDataFrame, then export to EPSG:4326 and EPSG:3857 GeoJSON via `data_io.export_hotspot_polygons()`.

In [ ]:
# Import modular export function
from src.data_io import export_hotspot_polygons
from config import HOTSPOT_EXPORT_CONFIG, PROJECT_ROOT

print("🔗 Combining all attributes...")
print("-" * 70)

# Merge landscape, LST, and population attributes with original GeoDataFrame
hotspot_polygons_enriched = hotspot_polygons_gdf.copy()

# Add landscape attributes
for col in landscape_df.columns:
    hotspot_polygons_enriched[col] = landscape_df[col].values

# Add LST statistics
for col in lst_stats_df.columns:
    hotspot_polygons_enriched[col] = lst_stats_df[col].values

# Add population statistics
for col in pop_stats_df.columns:
    hotspot_polygons_enriched[col] = pop_stats_df[col].values
hotspot_polygons_enriched['ghsl_year'] = ghsl_year

print(f"✓ Combined GeoDataFrame created")
print(f"  - Total columns: {len(hotspot_polygons_enriched.columns)}")
print(f"  - Total features: {len(hotspot_polygons_enriched):,}")
print(f"  - Original columns: {len(hotspot_polygons_gdf.columns)}")
print(f"  - Landscape columns: {len(landscape_df.columns)}")
print(f"  - LST columns: {len(lst_stats_df.columns)}")
print(f"  - Population columns: {len(pop_stats_df.columns)} + ghsl_year")

# Display column summary
print(f"\n📋 Column categories:")
print(f"  - Original: {list(hotspot_polygons_gdf.columns[:5])} ...")
print(f"  - Landscape: {list(landscape_df.columns[:5])} ...")
print(f"  - LST: {list(lst_stats_df.columns[:5])} ...")
print(f"  - Population: {list(pop_stats_df.columns)} + ['ghsl_year']")

# Export using modular function with corrected path structure
output_dir = HOTSPOT_EXPORT_CONFIG['output_base'] / HOTSPOT_EXPORT_CONFIG['polygons_subdir']

print(f"\n💾 Exporting GeoJSON files...")
print("-" * 70)

export_results = export_hotspot_polygons(
    gdf=hotspot_polygons_enriched,
    output_dir=output_dir,
    primary_crs=HOTSPOT_EXPORT_CONFIG['primary_crs'],
    web_crs=HOTSPOT_EXPORT_CONFIG['web_crs']
)

# Display results
print(f"✓ {export_results['primary']['crs']}: {export_results['primary']['path'].name}")
print(f"  - Size: {export_results['primary']['size_mb']:.2f} MB")
print(f"  - Use case: General GIS, web mapping, coordinate display")

print(f"\n✓ {export_results['web']['crs']}: {export_results['web']['path'].name}")
print(f"  - Size: {export_results['web']['size_mb']:.2f} MB")
print(f"  - Use case: Web maps, Google Maps, OpenStreetMap overlays")

print(f"\n📁 Export directory: {output_dir}")
print(f"   Total files: 2 GeoJSON")

print("\n" + "="*70)
print("✅ SECTION 4 COMPLETE: Comprehensive GeoDataFrame Developed")
print("="*70)
print(f"📊 Final dataset: {len(hotspot_polygons_enriched):,} polygons × {len(hotspot_polygons_enriched.columns)} attributes")
print(f"🗂️  Exported to: {output_dir.relative_to(PROJECT_ROOT)}")
print("="*70 + "\n")

### 4.6 Distribution Assessment

Diagnostic analysis of polygon counts across the four grouping dimensions (city, method, period, contour level). Coefficient of variation and skewness inform the choice of statistical approach for subsequent modelling in Notebook 06.

In [ ]:

print("📊 HOTSPOT POLYGON DISTRIBUTION ANALYSIS")
print("="*70 + "\n")

# Overall summary
print("📈 OVERALL SUMMARY:")
print("-" * 70)
print(f"Total polygons: {len(hotspot_polygons_enriched):,}")
print(f"Cities: {len(hotspot_polygons_enriched['city'].unique())}")
print(f"Methods: {', '.join(sorted(hotspot_polygons_enriched['method'].unique()))}")
print(f"Periods: {', '.join(sorted(hotspot_polygons_enriched['period'].unique()))}")
print(f"Contour levels: {sorted(hotspot_polygons_enriched['contour_level'].unique())}")

# UNIFIED TABLE: City × Method × Period × Contour
print("\n\n📊 COMPREHENSIVE DISTRIBUTION TABLE: City × Method × Period × Contour")
print("=" * 100)

# Group by all four dimensions
distribution_table = hotspot_polygons_enriched.groupby(
    ['city', 'method', 'period', 'contour_level']
).size().reset_index(name='n_polygons')

# Sort for better readability
distribution_table = distribution_table.sort_values(
    ['city', 'method', 'period', 'contour_level']
).reset_index(drop=True)

# Rename columns for clarity
distribution_table.columns = ['City', 'Method', 'Period', 'Contour', 'N_Polygons']

# Add contour interpretation column
contour_labels = {
    0.5: 'Broad (Less Critical)',
    0.75: 'Strong (Moderately Critical)',
    0.9: 'Core (Most Critical)'
}
distribution_table['Contour_Interpretation'] = distribution_table['Contour'].map(contour_labels)

# Display the comprehensive table
print(distribution_table.to_string(index=False))

# Summary statistics by grouping levels
print("\n\n📊 SUMMARY STATISTICS BY GROUPING LEVELS:")
print("=" * 100)

print("\n1. By City:")
city_summary = distribution_table.groupby('City')['N_Polygons'].agg(['sum', 'mean', 'std', 'min', 'max'])
city_summary.columns = ['Total', 'Mean', 'StdDev', 'Min', 'Max']
print(city_summary.to_string())

print("\n2. By Method:")
method_summary = distribution_table.groupby('Method')['N_Polygons'].agg(['sum', 'mean', 'std', 'min', 'max'])
method_summary.columns = ['Total', 'Mean', 'StdDev', 'Min', 'Max']
print(method_summary.to_string())

print("\n3. By Period:")
period_summary = distribution_table.groupby('Period')['N_Polygons'].agg(['sum', 'mean', 'std', 'min', 'max'])
period_summary.columns = ['Total', 'Mean', 'StdDev', 'Min', 'Max']
print(period_summary.to_string())

print("\n4. By Contour Level:")
contour_summary = distribution_table.groupby(['Contour', 'Contour_Interpretation'])['N_Polygons'].agg(['sum', 'mean', 'std', 'min', 'max'])
contour_summary.columns = ['Total', 'Mean', 'StdDev', 'Min', 'Max']
print(contour_summary.to_string())

# STATISTICAL ANALYSIS RECOMMENDATION
print("\n\n🔬 STATISTICAL ANALYSIS RECOMMENDATION:")
print("=" * 100)

# Calculate overall distribution statistics
overall_mean = distribution_table['N_Polygons'].mean()
overall_std = distribution_table['N_Polygons'].std()
overall_median = distribution_table['N_Polygons'].median()
overall_min = distribution_table['N_Polygons'].min()
overall_max = distribution_table['N_Polygons'].max()

cv = overall_std / overall_mean if overall_mean > 0 else 0
skewness = distribution_table['N_Polygons'].skew()

print(f"Overall Distribution Statistics:")
print(f"  - Mean: {overall_mean:.2f}")
print(f"  - Median: {overall_median:.0f}")
print(f"  - Std Dev: {overall_std:.2f}")
print(f"  - Min: {overall_min}")
print(f"  - Max: {overall_max}")
print(f"  - Range: {overall_max - overall_min}")
print(f"  - Coefficient of Variation (CV): {cv:.2f}")
print(f"  - Skewness: {skewness:.2f}")

print("\nRecommendation:")
if cv < 0.3 and abs(skewness) < 0.5:
    print("✓ LOW variability, SYMMETRIC distribution")
    print("  → SUITABLE for PARAMETRIC tests (hierarchical regression, ANOVA)")
    print("  → Proceed with your planned hierarchical regression from implementation guide")
elif cv < 0.5 and abs(skewness) < 1:
    print("⚠ MODERATE variability, SLIGHTLY skewed distribution")
    print("  → Consider ROBUST regression with M-estimators")
    print("  → Or log/sqrt transformation before hierarchical regression")
    print("  → Parametric tests still appropriate with caution")
else:
    print("⚠ HIGH variability or HIGHLY SKEWED distribution")
    print("  → Recommend NON-PARAMETRIC tests:")
    print("     • Kruskal-Wallis H-test (instead of ANOVA)")
    print("     • Mann-Whitney U-test (instead of t-test)")
    print("     • Or use Generalized Linear Mixed Models (GLMM) with Poisson/Negative Binomial")
    print("  → Alternative: Quantile regression or bootstrapped methods")

print("\nFor hierarchical/multilevel modeling:")
print(f"  - Level 1 (individual polygon): n = {len(hotspot_polygons_enriched):,}")
print(f"  - Level 2 (configuration): n = {len(distribution_table)} unique combinations")
print(f"  - Level 3 (city): n = {len(distribution_table['City'].unique())} cities")
print(f"  - Level 4 (method): n = {len(distribution_table['Method'].unique())} methods")
print(f"  - Cross-classified structure: {len(distribution_table)} city-method-period-contour combinations")

print("\n" + "="*100)
print("✅ DISTRIBUTION ASSESSMENT COMPLETE")
print("="*100 + "\n")

---

## Section 5: Stakeholder Survey Validation

**Objective:** Validate KDE-derived hotspot polygons against stakeholder-identified heat-stress locations to assess alignment between computational detection and local knowledge.

**Approach:**
- Focus on hotspots only (consistent with the project emphasis on urban warming)
- Simple polygon containment (`.within()`) with no additional buffers
- All detection methods combined per city-period configuration
- Stakeholder locations deduplicated by participant ID, city, and coordinates

**Metric:** Hotspot Hit Rate -- the percentage of stakeholder-reported hotspots falling within any KDE polygon. Interpretation thresholds: >= 70 % excellent, 50--70 % good, 30--50 % moderate, < 30 % poor.

### 5.1 Hit-Rate Analysis

Compute the stakeholder hit rate per city and period, summarise overall detection performance, and identify best- and worst-performing configurations.

In [ ]:
# ====================================================================
# SECTION 5.1: KDE POLYGON VALIDATION
# ====================================================================

"""
Validate KDE-derived hotspot polygons against stakeholder survey hotspots.
Uses simple polygon containment checks (no buffers, no confusion matrices).

Approach:
- Focus on HOTSPOTS only (project emphasis on warming/persistent heat)
- Check if stakeholder hotspots fall within ANY KDE polygon (all methods combined)
- Validation by City × Period (not by individual method)
- Uses entire KDE polygons (unary_union), not individual contours
- Deduplicates survey points by PID to ensure unique stakeholders
"""

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("=" * 80)
print("SECTION 5: STAKEHOLDER SURVEY VALIDATION (KDE POLYGON-BASED)")
print("=" * 80)

# ====================================================================
# STEP 1: PREPARE DATA
# ====================================================================

print("\n" + "-" * 80)
print("STEP 1: PREPARING VALIDATION DATA")
print("-" * 80)

# Load survey data (already in memory from previous sections)
try:
    # Check if survey data already loaded
    if 'survey_gdf' not in locals() or survey_gdf is None:
        from src.data_io import load_survey_data
        survey_gdf = load_survey_data()
        print("✓ Loaded survey data from file")
    else:
        print("✓ Using existing survey data in memory")
    
    # Display survey data structure for verification
    print(f"\nSurvey Data Structure:")
    print(f"  Columns: {survey_gdf.columns.tolist()}")
    print(f"  Shape: {survey_gdf.shape}")
    
    # Check for the spot type column
    if 'spot_type' in survey_gdf.columns:
        print(f"  Using 'spot_type' column")
        spot_column = 'spot_type'
    elif 'feature_spot' in survey_gdf.columns:
        print(f"  Using 'feature_spot' column")
        spot_column = 'feature_spot'
    else:
        raise ValueError(f"Cannot find spot type column. Available columns: {survey_gdf.columns.tolist()}")
    
    # Display unique values
    print(f"  Unique values in '{spot_column}': {survey_gdf[spot_column].unique()}")
    
    # Filter for HOTSPOTS only
    survey_hotspots = survey_gdf[
        survey_gdf[spot_column].astype(str).str.contains('hotspot', case=False, na=False)
    ].copy()
    
    print(f"\nSurvey Points Summary (BEFORE deduplication):")
    print(f"  Total survey points: {len(survey_gdf)}")
    print(f"  Hotspot entries: {len(survey_hotspots)}")
    
    # CRITICAL: Deduplicate by PID (participant ID) and location
    # The master_composite.csv can have multiple rows per person
    if 'pid' in survey_hotspots.columns:
        print(f"\n  Unique PIDs (stakeholders): {survey_hotspots['pid'].nunique()}")
        
        # Deduplicate: Keep unique combinations of pid + city + lat + lon
        # This ensures each stakeholder's hotspot location is counted once
        if 'city' in survey_hotspots.columns:
            dedup_cols = ['pid', 'city']
            # Add geometry-based deduplication
            survey_hotspots['lat_round'] = survey_hotspots.geometry.y.round(4)
            survey_hotspots['lon_round'] = survey_hotspots.geometry.x.round(4)
            dedup_cols.extend(['lat_round', 'lon_round'])
            
            survey_hotspots_unique = survey_hotspots.drop_duplicates(subset=dedup_cols).copy()
            survey_hotspots_unique = survey_hotspots_unique.drop(columns=['lat_round', 'lon_round'])
            
            print(f"\n  Deduplication: {len(survey_hotspots)} → {len(survey_hotspots_unique)} unique hotspot locations")
            print(f"  Removed {len(survey_hotspots) - len(survey_hotspots_unique)} duplicate entries")
            
            survey_hotspots = survey_hotspots_unique
        else:
            print(f"  ⚠ 'city' column not found, deduplication by PID + geometry only")
            survey_hotspots['lat_round'] = survey_hotspots.geometry.y.round(4)
            survey_hotspots['lon_round'] = survey_hotspots.geometry.x.round(4)
            survey_hotspots_unique = survey_hotspots.drop_duplicates(subset=['pid', 'lat_round', 'lon_round']).copy()
            survey_hotspots_unique = survey_hotspots_unique.drop(columns=['lat_round', 'lon_round'])
            survey_hotspots = survey_hotspots_unique
    else:
        print(f"  ⚠ 'pid' column not found - cannot deduplicate by stakeholder")
    
    print(f"\nFINAL Survey Points (AFTER deduplication):")
    print(f"  Unique hotspot locations: {len(survey_hotspots)}")
    
    # Display breakdown by city if column exists
    if 'city' in survey_hotspots.columns:
        print(f"\n  Hotspots by city:")
        hotspot_city_counts = survey_hotspots['city'].value_counts()
        for city, count in hotspot_city_counts.items():
            print(f"    {city}: {count}")
    
    # Check CRS alignment
    if hotspot_polygons_enriched.crs != survey_gdf.crs:
        print(f"\n⚠ CRS mismatch detected - reprojecting survey data")
        print(f"  Hotspot polygons CRS: {hotspot_polygons_enriched.crs}")
        print(f"  Survey data CRS: {survey_gdf.crs}")
        survey_hotspots = survey_hotspots.to_crs(hotspot_polygons_enriched.crs)
        print(f"  ✓ Survey data reprojected to {hotspot_polygons_enriched.crs}")
    else:
        print(f"\n✓ CRS alignment OK: {survey_gdf.crs}")
    
except Exception as e:
    print(f"\n✗ Error loading survey data: {e}")
    print(f"\nAttempting to diagnose issue...")
    if 'survey_gdf' in locals():
        print(f"  survey_gdf columns: {survey_gdf.columns.tolist()}")
        print(f"  survey_gdf shape: {survey_gdf.shape}")
        if 'spot_type' in survey_gdf.columns:
            print(f"  spot_type unique values: {survey_gdf['spot_type'].unique()}")
            print(f"  spot_type value counts:\n{survey_gdf['spot_type'].value_counts()}")
    raise

# Check hotspot polygons availability
print(f"\nHotspot Polygons Summary:")
print(f"  Total polygons: {len(hotspot_polygons_enriched)}")
print(f"  Cities: {hotspot_polygons_enriched['city'].nunique()}")
print(f"  Methods: {', '.join(sorted(hotspot_polygons_enriched['method'].unique()))}")
print(f"  Periods: {', '.join(sorted(hotspot_polygons_enriched['period'].unique()))}")

# ====================================================================
# STEP 2: VALIDATE BY CITY × PERIOD (ALL METHODS COMBINED)
# ====================================================================

print("\n" + "-" * 80)
print("STEP 2: VALIDATING HOTSPOT DETECTION BY CITY × PERIOD")
print("NOTE: Checking if stakeholder hotspots fall within ANY KDE polygon")
print("      (all methods combined: P90, GI_STAR, CONFIRMED)")
print("-" * 80)

validation_results = []

# Get unique combinations
cities = sorted(hotspot_polygons_enriched['city'].unique())
periods = ['Day', 'Night', 'Persistent']

for city in cities:
    print(f"\n[{city}]")
    
    # Filter survey hotspots for this city
    city_hotspots = survey_hotspots[survey_hotspots['city'] == city].copy()
    
    if len(city_hotspots) == 0:
        print(f"  ⚠ No survey hotspots for {city}")
        continue
    
    print(f"  Survey hotspots: {len(city_hotspots)}")
    
    for period in periods:
        # Filter hotspot polygons for this city-period (ALL METHODS)
        config_filter = (
            (hotspot_polygons_enriched['city'] == city) & 
            (hotspot_polygons_enriched['period'] == period)
        )
        config_polygons = hotspot_polygons_enriched[config_filter].copy()
        
        if len(config_polygons) == 0:
            print(f"    {period}: No KDE polygons (NA)")
            # Store result with NA values
            validation_results.append({
                'City': city,
                'Period': period,
                'N_Survey_Hotspots': len(city_hotspots),
                'N_KDE_Polygons': 'NA',
                'Methods_Contributing': 'NA',
                'Hotspot_Hits': 'NA',
                'Hotspot_Hit_Rate_%': 'NA'
            })
            continue
        
        # Get list of methods that contributed polygons
        methods_contributing = sorted(config_polygons['method'].unique())
        
        # Merge ALL polygons for this city-period into one union
        # NOTE: Uses entire KDE polygon extent (unary_union), not individual contours
        config_union = config_polygons.geometry.unary_union
        
        # Count hotspot hits (hotspots WITHIN ANY KDE polygon)
        hotspot_hits = 0
        for idx, point in city_hotspots.iterrows():
            if point.geometry.within(config_union):
                hotspot_hits += 1
        
        # Calculate hit rate
        hotspot_hit_rate = (hotspot_hits / len(city_hotspots) * 100) if len(city_hotspots) > 0 else 0
        
        # Store results
        validation_results.append({
            'City': city,
            'Period': period,
            'N_Survey_Hotspots': len(city_hotspots),
            'N_KDE_Polygons': len(config_polygons),
            'Methods_Contributing': '+'.join(methods_contributing),
            'Hotspot_Hits': hotspot_hits,
            'Hotspot_Hit_Rate_%': round(hotspot_hit_rate, 1)
        })
        
        print(f"    {period}: Hit Rate={hotspot_hit_rate:.1f}% ({hotspot_hits}/{len(city_hotspots)}) - {'+'.join(methods_contributing)}")

# ====================================================================
# STEP 3: CREATE VALIDATION SUMMARY TABLE
# ====================================================================

print("\n" + "-" * 80)
print("STEP 3: VALIDATION SUMMARY TABLE")
print("-" * 80 + "\n")

validation_summary_df = pd.DataFrame(validation_results)

# Display full table
print("FULL VALIDATION RESULTS (City × Period):")
print("=" * 100)
print(validation_summary_df.to_string(index=False))

# ====================================================================
# STEP 4: SUMMARY STATISTICS
# ====================================================================

print("\n" + "-" * 80)
print("STEP 4: SUMMARY STATISTICS (Excluding NA Configurations)")
print("-" * 80)

# Filter out NA configurations for statistics
valid_configs = validation_summary_df[
    validation_summary_df['N_KDE_Polygons'] != 'NA'
].copy()

if len(valid_configs) > 0:
    print(f"\nTotal configurations validated: {len(valid_configs)} (out of {len(validation_summary_df)})")
    print(f"Configurations with NA: {len(validation_summary_df) - len(valid_configs)}")
    
    # Overall statistics
    print(f"\n1. OVERALL PERFORMANCE:")
    print(f"   Average Hotspot Hit Rate: {valid_configs['Hotspot_Hit_Rate_%'].mean():.1f}%")
    print(f"   Median Hotspot Hit Rate: {valid_configs['Hotspot_Hit_Rate_%'].median():.1f}%")
    print(f"   Std Dev: {valid_configs['Hotspot_Hit_Rate_%'].std():.1f}%")
    print(f"   Min: {valid_configs['Hotspot_Hit_Rate_%'].min():.1f}%")
    print(f"   Max: {valid_configs['Hotspot_Hit_Rate_%'].max():.1f}%")
    
    # By Period
    print(f"\n2. PERFORMANCE BY PERIOD:")
    period_stats = valid_configs.groupby('Period')['Hotspot_Hit_Rate_%'].agg(['mean', 'std', 'min', 'max', 'count'])
    period_stats.columns = ['Mean_%', 'StdDev_%', 'Min_%', 'Max_%', 'N_Cities']
    print(period_stats.to_string())
    
    # By City
    print(f"\n3. PERFORMANCE BY CITY:")
    city_stats = valid_configs.groupby('City')['Hotspot_Hit_Rate_%'].agg(['mean', 'std', 'min', 'max', 'count'])
    city_stats.columns = ['Mean_%', 'StdDev_%', 'Min_%', 'Max_%', 'N_Periods']
    print(city_stats.to_string())
    
    # Best and worst performing configurations
    best_config = valid_configs.loc[valid_configs['Hotspot_Hit_Rate_%'].idxmax()]
    worst_config = valid_configs.loc[valid_configs['Hotspot_Hit_Rate_%'].idxmin()]
    
    print(f"\n4. BEST PERFORMING CONFIGURATION:")
    print(f"   City: {best_config['City']}")
    print(f"   Period: {best_config['Period']}")
    print(f"   Hit Rate: {best_config['Hotspot_Hit_Rate_%']}%")
    print(f"   Hotspots Captured: {best_config['Hotspot_Hits']}/{best_config['N_Survey_Hotspots']}")
    print(f"   Methods: {best_config['Methods_Contributing']}")
    
    print(f"\n5. WORST PERFORMING CONFIGURATION:")
    print(f"   City: {worst_config['City']}")
    print(f"   Period: {worst_config['Period']}")
    print(f"   Hit Rate: {worst_config['Hotspot_Hit_Rate_%']}%")
    print(f"   Hotspots Captured: {worst_config['Hotspot_Hits']}/{worst_config['N_Survey_Hotspots']}")
    print(f"   Methods: {worst_config['Methods_Contributing']}")
    
else:
    print("\n⚠ No valid configurations found (all are NA)")

# ====================================================================
# STEP 5: INTERPRETATION GUIDELINES
# ====================================================================

print("\n" + "-" * 80)
print("INTERPRETATION GUIDELINES")
print("-" * 80)

print("""
Hit Rate Interpretation:
  ≥ 70%    EXCELLENT - High correspondence between KDE polygons and stakeholder knowledge
  50-70%   GOOD      - Moderate correspondence, KDE captures most stakeholder-identified areas
  30-50%   MODERATE  - Partial correspondence, some stakeholder hotspots not captured
  < 30%    POOR      - Low correspondence, significant disagreement

Validation Approach:
- Focus on HOTSPOTS only (project emphasis on urban warming/persistent heat)
- Combined all detection methods (P90, GI_STAR, CONFIRMED) per city-period
- Stakeholder hotspots validated against ANY detected KDE polygon
- Deduplication by PID ensures unique stakeholder contributions
- This tests if stakeholder knowledge aligns with the overall hotspot detection framework

Notes:
- Hit Rate: % of unique stakeholder hotspots that fall within ANY KDE polygon
- Higher hit rate indicates better alignment between stakeholder knowledge and detection methods
- NA configurations: No KDE polygons detected for that city-period combination
- Methods_Contributing: Shows which methods detected polygons (P90, GI_STAR, CONFIRMED)
- Coolspots excluded from evaluation but retained for visualization purposes
""")

print("\n" + "=" * 80)
print("✅ SECTION 5 COMPLETE: HOTSPOT VALIDATION ANALYSIS FINISHED")
print("=" * 80 + "\n")

---

## Section 6: Visualisation

All plotting functions are modularised in `src/visualization.py` with built-in skip logic. Outputs are saved to `data/outputs/05_hotspot_analysis/maps/` (maps) and `data/outputs/05_hotspot_analysis/figures/` (charts). A canonical 25-colour palette for the Cluster25 landscape classes is defined in `config.CLUSTER25_COLORS`.

### 6.1 City Hotspot Grid Maps

One figure per city with a 1 x 3 layout (Day | Night | Persistent). Each panel shows:

- **Base layer:** Cluster25 physiographic landscape (canonical colours, semi-transparent fill)
- **OSM features:** Roads (thin grey lines), waterbodies (light blue fill)
- **City boundaries:** Main AOI (solid black), adjacent cities (dashed grey)
- **Hotspot overlays:** P90 (orange) and Gi\* (yellow-orange gradient) KDE polygons at 50/75/90 % contours
- **Survey points:** Stakeholder hotspots (red stars) and coolspots (blue circles)
- **Cartographic elements:** Scale bar, north arrow, hotspot area annotation

Exported as PDF (vector) and PNG (300 DPI) per city.

### 6.2 Gi\* Distribution Plots

One figure per city with two subplots (Day, Night). Each subplot shows the histogram of Gi\* Z-scores with a theoretical $N(0,1)$ overlay, significance threshold lines at $\pm 1.96$ and $\pm 2.58$, and shaded hot/cold regions. Summary statistics (mean, std, skewness, kurtosis, exceedance percentages) are annotated.

### 6.3 Hotspot Area by Contour Level and Cluster25

A single multi-panel figure with three vertically stacked rows (Day, Night, Persistent). Within each row, cities are arranged along the x-axis, grouped by method (P90, Gi\*) and contour level (90 %, 75 %, 50 %). Each bar is a stacked composition of Cluster25 landscape classes using canonical colours, with patch counts annotated above.

Two supporting CSV tables are exported alongside the chart:

| Table | Content |
|---|---|
| Table 1 | Hotspot characterisation: city, method, period, contour, polygon count, total area, mean area, dominant Cluster25 class and its proportion |
| Table 2 | Cluster25 colour reference: cluster ID, hex colour, climate class, geology class, elevation class, description |

### 6.4 Population Exposure by Period, Method & Contour Level

Half-donut cross-section charts (lithosphere-style) — one per city — showing **population exposure inside UHI hotspot polygons**.  Each chart uses split halves:

- **Left half (Day ☀):** Warm orange tones, three nested rings showing Period → Method (P90/Gi\*) → Contour (90%/75%/50%)
- **Right half (Night ☾):** Cool blue tones, same hierarchical breakdown
- **Labels:** Elbow callout lines with population values on the flat cut edge

This stratified design allows rapid comparison of daytime vs. nighttime exposure across different detection methods and hotspot intensity levels.

### 6.5 National Summary Map — Cluster25 Landscape Context

A single-page map of Nigeria showing all 28 workshop cities overlaid on the Cluster25 physiographic landscape base. Each city is represented by a **proportional circle** sized by daytime population exposure, with a smaller inner circle for nighttime exposure. The map includes:

- State boundaries (grey lines)
- Landscape class legend (compact grid with 25 colour swatches)
- Persistent-heat cities highlighted with bold boundary rings
- Gi\* hotspot cities marked with star symbols
- City labels positioned to minimise overlap
- Textbox with exposure statistics (total day/night population, Gi\* area, etc.)

This map provides the spatial context for hotspot distribution across Nigeria's diverse landscape types.

### 6.6 Stakeholder Perception by Landscape Character (Cluster25)

Stakeholder workshop responses (from **master_composite.csv**) are spatially linked to the Cluster25 landscape classification via city-based assignment. Each of the 28 workshop cities is mapped to its dominant landscape class, ensuring **all 115 stakeholders** are included (not just those with spot coordinates). This section produces:

1. **Reasons for Urban Heating vs. Suggested Mitigation** — Side-by-side horizontal stacked-bar chart with CL-{id} labels on y-axis, showing stakeholder perceptions grouped by landscape type. Includes a textbox listing which cities belong to each cluster. Both panels use "No. of Stakeholders" labels and show bold counts (not "n=X").

2. **Tree Species Suggested by Stakeholders** — Four heatmaps with light-to-dark green gradients and bold annotations:
   - **Heatmap 2a:** Top 15 species × Cluster25 landscape classes
   - **Heatmap 2b:** ALL 34 species × Cluster25 landscape classes  
   - **Heatmap 2c:** Top 15 species × 28 cities
   - **Heatmap 2d:** ALL 34 species × 28 cities

   Each cell shows the number of **unique stakeholders** who suggested that species (deduplicated by PID + species + group to avoid inflating counts from multiple survey spots per stakeholder). Heatmaps 2c & 2d use rotated city labels for readability.

### Section 6.1 -- Generate City Hotspot Grid Maps

In [ ]:
# =============================================================================
# Section 6.1: 1×3 Hotspot Grid Maps per City (Sequential with tqdm)
# =============================================================================
# Create comprehensive visualization showing P90 and Gi* overlayed on same maps
# Layout: 1 row × 3 periods (columns) = 3 panels per city
# Uses modular function with skip logic for efficiency

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
from tqdm import tqdm
import time
import warnings
import importlib

# Suppress OGR field type warnings
warnings.filterwarnings('ignore', message='.*unsupported OGR type.*')

# Reload visualization module to get latest changes
import visualization
importlib.reload(visualization)
from visualization import create_city_hotspot_grid_maps

print("="*80)
print("SECTION 6.1: 1×3 HOTSPOT GRID MAPS (Overlayed Methods)")
print("="*80)

# -----------------------------------------------------------------------------
# 6.1.1: Setup and Configuration
# -----------------------------------------------------------------------------

# Test with one city first
TEST_CITY = None  # Change to 'CityName' to process only one city, None for all

# Output directory
viz_output_dir = OUTPUT_DIR / '05_hotspot_analysis'
maps_dir = viz_output_dir / 'maps'
maps_dir.mkdir(parents=True, exist_ok=True)

print("\n✓ Configuration loaded")
print(f"  - Testing with city: {'ALL' if TEST_CITY is None else TEST_CITY}")
print(f"  - Grid layout: 1 row × 3 periods = 3 panels (P90 & Gi* overlayed)")
print(f"  - Output directory: {maps_dir}")

# -----------------------------------------------------------------------------
# 6.1.2: Load Base Layers
# -----------------------------------------------------------------------------

print("\n" + "="*80)
print("Loading base layers...")
print("="*80)

# Use landscape data already loaded in previous cells
if 'landscape_gdf_4326' in dir():
    print(f"✓ Using landscape data from memory: {len(landscape_gdf_4326)} features")
else:
    print("✗ Landscape data not found in memory. Please run previous cells first.")

# -----------------------------------------------------------------------------
# 6.1.3: Generate Maps with Skip Logic
# -----------------------------------------------------------------------------

print("\n" + "="*80)
print("Generating 1×3 hotspot grid maps with skip logic...")
print("="*80)

# Determine which cities to process
cities_to_process = [TEST_CITY] if TEST_CITY else CITIES

# Generate maps with progress bar
start_time = time.time()
results = []

for city in tqdm(cities_to_process, desc="Processing cities", unit="city"):
    try:
        result = create_city_hotspot_grid_maps(
            city_name=city,
            hotspot_polygons=hotspot_polygons_enriched,
            survey_points=survey_gdf,
            landscape_data=landscape_gdf_4326,
            output_dir=maps_dir,
            show_plot=False,
            skip_if_exists=True  # Enable skip logic
        )
        if result:
            results.append(result)
            if result.get('skipped', False):
                tqdm.write(f"⏭  Skipped: {city} (maps already exist with current parameters)")
            else:
                tqdm.write(f"✓ Completed: {city}")
        else:
            tqdm.write(f"✗ Failed: {city}")
    except Exception as e:
        tqdm.write(f"✗ Error for {city}: {e}")
        import traceback
        traceback.print_exc()

end_time = time.time()
elapsed_time = end_time - start_time

# Print summary
print("\n" + "="*80)
print("SECTION 6.1 COMPLETE")
print("="*80)
print(f"✓ Processed {len(results)}/{len(cities_to_process)} cities successfully")

# Count skipped and generated
skipped_count = sum(1 for r in results if r.get('skipped', False))
generated_count = len(results) - skipped_count

print(f"  - Generated: {generated_count} cities")
print(f"  - Skipped: {skipped_count} cities (already exist)")
print(f"✓ Total time: {elapsed_time:.1f} seconds ({elapsed_time/60:.2f} minutes)")
if results:
    print(f"✓ Average time per city: {elapsed_time/len(results):.1f} seconds")
print(f"✓ Maps saved to: {maps_dir}")

# List generated/skipped files
if results:
    print("\nSummary:")
    for result in results:
        status = "SKIPPED" if result.get('skipped', False) else "GENERATED"
        print(f"  {status}: {result['city']}")

### Section 6.2 -- Gi\* Z-Score Distribution Plots

In [ ]:
# =============================================================================
# Section 6.2: Gi* Distribution Plots per City
# =============================================================================
# Visualize statistical distributions of Gi* Z-scores to validate hotspot 
# detection thresholds
# Layout: 1 row × 2 periods = 2 subplots per city (Day and Night only)
# Uses modular function with skip logic for efficiency

import importlib

# Reload visualization module to get latest changes
importlib.reload(visualization)
from visualization import create_gi_star_distribution_plots

print("="*80)
print("SECTION 6.2: Gi* DISTRIBUTION PLOTS")
print("="*80)

# Test with one city first
TEST_CITY = None  # Change to 'CityName' to process only one city, None for all

# Output directory for figures (not maps)
figures_dir = viz_output_dir / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

print("\n✓ Configuration loaded")
print(f"  - Testing with city: {'ALL' if TEST_CITY is None else TEST_CITY}")
print(f"  - Layout: 1 row × 2 periods = 2 subplots per city (Day and Night)")
print(f"  - Output directory: {figures_dir}")

print("\n" + "="*80)
print("Generating Gi* distribution plots with skip logic...")
print("="*80)

# Determine which cities to process
cities_to_process = [TEST_CITY] if TEST_CITY else CITIES

# Generate plots with progress bar
start_time = time.time()
results = []

for city in tqdm(cities_to_process, desc="Processing cities", unit="city"):
    try:
        result = create_gi_star_distribution_plots(
            city_name=city,
            output_dir=figures_dir,
            skip_if_exists=True  # Enable skip logic
        )
        if result:
            results.append(result)
            if result.get('skipped', False):
                tqdm.write(f"⏭  Skipped: {city} (plots already exist)")
            else:
                tqdm.write(f"✓ Completed: {city}")
        else:
            tqdm.write(f"✗ Failed: {city}")
    except Exception as e:
        tqdm.write(f"✗ Error for {city}: {e}")
        import traceback
        traceback.print_exc()

end_time = time.time()
elapsed_time = end_time - start_time

# Print summary
print("\n" + "="*80)
print("SECTION 6.2 COMPLETE")
print("="*80)
print(f"✓ Processed {len(results)}/{len(cities_to_process)} cities successfully")

# Count skipped and generated
skipped_count = sum(1 for r in results if r.get('skipped', False))
generated_count = len(results) - skipped_count

print(f"  - Generated: {generated_count} cities")
print(f"  - Skipped: {skipped_count} cities (already exist)")
print(f"✓ Total time: {elapsed_time:.1f} seconds ({elapsed_time/60:.2f} minutes)")
if results:
    print(f"✓ Average time per city: {elapsed_time/len(results):.1f} seconds")
print(f"✓ Plots saved to: {figures_dir}")

# List generated/skipped files
if results:
    print("\nSummary:")
    for result in results:
        status = "SKIPPED" if result.get('skipped', False) else "GENERATED"
        print(f"  {status}: {result['city']}")

### Section 6.3 -- Hotspot Area by Contour Level and Cluster25

In [ ]:
# Reload visualization module to get latest function
import importlib
import sys

# Remove all cached visualization modules
modules_to_remove = [k for k in sys.modules.keys() if 'visualization' in k]
for mod in modules_to_remove:
    del sys.modules[mod]

# Import fresh
from src.visualization import create_hotspot_area_by_contour_plots

# Create hotspot area by contour plots (2 methods: P90 and GI_STAR only)
print("\n" + "="*80)
print("SECTION 6.3: HOTSPOT AREA BY CONTOUR LEVEL & CLUSTER25")
print("="*80 + "\n")

print("Creating grouped stacked bar charts showing hotspot area distribution by:")
print("  - Contour levels: 90% (Most Critical), 75% (Moderately Critical), 50% (Less Critical)")
print("  - Methods: P90, GI_STAR (NO CONFIRMED)")
print("  - Landscape composition: Stacked by 25 unique Cluster25 colors")
print("  - Fragmentation: Patch counts annotated on each bar")
print("\n")

# Create plots
result = create_hotspot_area_by_contour_plots(
    hotspot_polygons_enriched=hotspot_polygons_enriched,
    landscape_gdf_4326=landscape_gdf_4326,
    output_dir=figures_dir,
    skip_if_exists=True
)

if result and result['success']:
    if result['skipped']:
        print(f"✓ Plots already exist, skipped")
    else:
        print(f"\n✓ Successfully created hotspot area by contour plots!")
    
    print(f"\nOutputs:")
    print(f"  📊 Chart (PDF): {result['pdf'].name}")
    print(f"  📊 Chart (PNG): {result['png'].name}")
    print(f"  📄 Table 1 (Characterization): {result['table1'].name}")
    print(f"  📄 Table 2 (Cluster25 Colors): {result['table2'].name}")
    
    # Display characterization table summary
    import pandas as pd
    table1 = pd.read_csv(result['table1'])
    print(f"\n📋 Table 1 Summary: {len(table1)} rows")
    print(f"   Cities: {table1['City'].nunique()}")
    print(f"   Methods: {', '.join(table1['Method'].unique())}")
    print(f"   Contours: {', '.join(map(str, sorted(table1['Contour'].unique(), reverse=True)))}")
    
    # Display Cluster25 table summary
    table2 = pd.read_csv(result['table2'])
    print(f"\n🎨 Table 2 Summary: {len(table2)} Cluster25 classes with colors")
    print(f"   Climate types: {table2['Climate_Class'].nunique()}")
    print(f"   Geology types: {table2['Geology_Class'].nunique()}")
else:
    print(f"✗ Failed to create hotspot area by contour plots")

print("\n" + "="*80)
print("END SECTION 6.3")
print("="*80)

### Section 6.4 — Population Exposure by Period, Method & Contour Level

Half-donut cross-section charts (lithosphere-style) — one per city — showing **how much population falls inside UHI hotspot polygons**:

| Layer | Level | Left half (Day ☀) | Right half (Night ☾) | Colour rule |
|-------|-------|--------------------|----------------------|-------------|
| Ring 1 (inner) | **Period** | Day total | Night total | Full-saturation primary |
| Ring 2 (middle) | **Method** | P90 / Gi\* | P90 / Gi\* | ~35 % faded |
| Ring 3 (outer) | **Contour** | 90 % / 75 % / 50 % | 90 % / 75 % / 50 % | ~60 % faded |

Population values are labelled on the flat cut edge with horizontal leader lines and colour swatches — every segment is readable.
Only `P90` and `GI_STAR` methods × `Day` and `Night` periods are included (no `VALIDATED` or `Persistent`).

In [ ]:
# Reload visualization module to get latest function
import importlib
import sys

modules_to_remove = [k for k in sys.modules.keys() if 'visualization' in k]
for mod in modules_to_remove:
    del sys.modules[mod]

from src.visualization import create_population_exposure_donuts

# Create per-city half-donut cross-section charts
print("\n" + "="*80)
print("SECTION 6.4: POPULATION EXPOSURE HALF-DONUT CROSS-SECTION")
print("="*80 + "\n")

print("Creating separate half-donut charts per city:")
print("  Left half:   Day ☀  (warm orange tones)")
print("  Right half:  Night ☾ (cool blue tones)")
print("  Ring 1:      Period totals")
print("  Ring 2:      Method — P90 / Gi*")
print("  Ring 3:      Contour — 90% / 75% / 50%")
print("  Labels:      Elbow callout lines with hierarchical grouping")
print()

result = create_population_exposure_donuts(
    hotspot_polygons_enriched=hotspot_polygons_enriched,
    output_dir=figures_dir,
    skip_if_exists=False,
)

if result and result.get('success'):
    if result.get('skipped'):
        print("✓ All plots already exist, skipped")
    else:
        print("\n✓ Successfully created per-city half-donut charts!")

    if 'cities' in result and isinstance(result['cities'], dict):
        print(f"\nOutputs ({len(result['cities'])} cities):")
        for city, paths in result['cities'].items():
            print(f"  📊 {city}: {paths['pdf'].name} / {paths['png'].name}")
else:
    print("✗ Failed to create half-donut charts")

print("\n" + "="*80)
print("END SECTION 6.4")
print("="*80)

## 6.5: National Summary Map — Proportional Symbol Map

Create a single-page map showing all five study cities on the Nigeria base map with proportional symbols sized by total population exposure and split by Day/Night periods.

In [ ]:
# =============================================================================
# Section 6.5: National Summary Map with Cluster25 Base Layer
# =============================================================================

from src.visualization import create_national_summary_map
import geopandas as gpd
import config

print("="*80)
print("SECTION 6.5: NATIONAL SUMMARY MAP (Cluster25 Base)")
print("="*80)

# -----------------------------------------------------------------------------
# Load data
# -----------------------------------------------------------------------------

print("\nLoading data...")

# Load hotspot polygons
hotspots_path = OUTPUT_DIR / '05_hotspot_analysis' / 'hotspot_polygons' / 'hotspot_polygons_EPSG4326.geojson'
hotspots_gdf = gpd.read_file(hotspots_path)
print(f"✓ Loaded {len(hotspots_gdf)} hotspot polygons")

# Use landscape data from memory (or load if needed)
if 'landscape_gdf_4326' in dir():
    landscape_gdf = landscape_gdf_4326.copy()
    print(f"✓ Using landscape data from memory: {len(landscape_gdf)} features")
else:
    landscape_path = LANDSCAPE_DATA_PATH
    landscape_gdf = gpd.read_file(landscape_path)
    print(f"✓ Loaded landscape data: {len(landscape_gdf)} features")

# Load Nigeria boundary
boundary_path = PROJECT_ROOT / 'data' / 'inputs' / 'geospatial_features' / 'boundary_Nigeria.geojson'
nigeria_gdf = gpd.read_file(boundary_path)
print(f"✓ Loaded Nigeria boundary")

# -----------------------------------------------------------------------------
# Create national summary map
# -----------------------------------------------------------------------------

print("\nCreating national summary map...")

result = create_national_summary_map(
    hotspots_gdf=hotspots_gdf,
    landscape_gdf=landscape_gdf,
    nigeria_gdf=nigeria_gdf,
    cluster25_colors=config.CLUSTER25_COLORS,
    output_dir=viz_output_dir,
    figsize=(10, 10),
    day_color='#FF9500',      # Orange
    night_color='#4A4A4A',    # Dark grey
    scale_factor=1.0,
    show_plot=True,
)

print(f"\n✓ Map saved to: {result['png_path'].parent.name}/")
print(f"  - {result['n_cities']} cities analyzed")
print(f"  - {result['n_hotspots']} hotspots ({result['total_area_km2']:.1f} km²)")
print(f"  - Day: {result['total_day_pop']:,.0f} | Night: {result['total_night_pop']:,.0f} persons exposed")
print(f"  - Gi* significant: {result['n_gi_star']} hotspots ({result['gi_star_area_km2']:.1f} km²)")

## 6.6: Stakeholder Perception by Landscape Character (Cluster25)

This section links stakeholder workshop responses (from **master_composite.csv**) to the Cluster25 landscape classification via city-based assignment. Each of the 28 workshop cities is mapped to its dominant landscape class using spatial joins of stakeholder spot locations, with manual fallbacks for 8 cities where no stakeholders recorded spot coordinates.

**Coverage:** All 115 workshop stakeholders are included — 96 provided reasons for urban heating, 113 suggested mitigation strategies, and 45 suggested tree species for urban cooling.

**Outputs:**
- 1 reasons/mitigation chart (side-by-side stacked bars by Cluster25, with city breakdown textbox)
- 4 tree species heatmaps (top-15/all × cluster25/city combinations, deduped by unique stakeholder)

In [ ]:
# =============================================================================
# Section 6.6: Stakeholder Perception by Landscape Character (Cluster25)
# =============================================================================

from src.visualization import (
    create_reasons_mitigation_chart,
    create_tree_species_heatmap,
)
import config

print("=" * 80)
print("SECTION 6.6: STAKEHOLDER PERCEPTION BY LANDSCAPE CHARACTER (Cluster25)")
print("=" * 80)

# ── Load data ────────────────────────────────────────────────────────────────

print("\n[1/6] Loading stakeholder survey data...")
survey_df = pd.read_csv(config.SURVEY_DATA_PATH)
print(f"✓ Loaded {len(survey_df)} records from master_composite.csv")

workshop_df = survey_df[survey_df['survey status'] == 'Yes']
n_stakeholders = workshop_df['pid'].nunique()
n_cities = workshop_df['matched_city_y'].nunique()
print(f"  → {n_stakeholders} workshop stakeholders across {n_cities} cities")

print("\n[2/6] Loading landscape data...")
landscape_gdf = gpd.read_file(config.LANDSCAPE_DATA_PATH)
print(f"✓ Loaded {len(landscape_gdf)} landscape polygons (CRS: {landscape_gdf.crs})")

# ── Chart 1: Reasons vs. Mitigation ─────────────────────────────────────────

print("\n[3/6] Creating Reasons vs. Mitigation chart...")
result_rm = create_reasons_mitigation_chart(
    survey_df=survey_df,
    landscape_gdf=landscape_gdf,
    output_dir=viz_output_dir,
    cluster25_colors=config.CLUSTER25_COLORS,
    figsize=(18, 9),
    show_plot=True,
)

if result_rm.get('success'):
    print(f"\n✓ Chart 1 saved: {result_rm['png_path'].name}")
    print(f"  → {result_rm['n_reason_stakeholders']} stk. gave reasons, "
          f"{result_rm['n_mitig_stakeholders']} stk. gave mitigation")
    print(f"  → {result_rm['n_cities']} cities, "
          f"{result_rm['n_cluster25_classes']} landscape classes")

# ── Chart 2a: Top 15 Tree Species × Cluster25 ───────────────────────────────

print("\n[4a/6] Creating Tree Species heatmap — Top 15 × Cluster25...")
result_ts1 = create_tree_species_heatmap(
    survey_df=survey_df,
    landscape_gdf=landscape_gdf,
    output_dir=viz_output_dir,
    top_n_species=15,
    group_by='cluster25',
    show_plot=True,
)

if result_ts1.get('success'):
    print(f"\n✓ Chart 2a saved: {result_ts1['png_path'].name}")
    print(f"  → {result_ts1['n_stakeholders']} stakeholders, "
          f"{result_ts1['n_cities']} cities")
    print(f"  → {result_ts1['n_species']} top species, "
          f"{result_ts1['n_groups']} landscape classes")
    print(f"  → Total tree mentions: {result_ts1['total_mentions']}")

# ── Chart 2b: ALL Tree Species × Cluster25 ──────────────────────────────────

print("\n[4b/6] Creating Tree Species heatmap — ALL species × Cluster25...")
result_ts2 = create_tree_species_heatmap(
    survey_df=survey_df,
    landscape_gdf=landscape_gdf,
    output_dir=viz_output_dir,
    top_n_species=None,
    group_by='cluster25',
    show_plot=True,
)

if result_ts2.get('success'):
    print(f"\n✓ Chart 2b saved: {result_ts2['png_path'].name}")
    print(f"  → {result_ts2['n_stakeholders']} stakeholders, "
          f"{result_ts2['n_cities']} cities")
    print(f"  → All {result_ts2['n_species']} species, "
          f"{result_ts2['n_groups']} landscape classes")
    print(f"  → Total tree mentions: {result_ts2['total_mentions']}")

# ── Chart 2c: Top 15 Tree Species × Cities ──────────────────────────────────

print("\n[5a/6] Creating Tree Species heatmap — Top 15 × Cities...")
result_ts3 = create_tree_species_heatmap(
    survey_df=survey_df,
    landscape_gdf=landscape_gdf,
    output_dir=viz_output_dir,
    top_n_species=15,
    group_by='city',
    show_plot=True,
)

if result_ts3.get('success'):
    print(f"\n✓ Chart 2c saved: {result_ts3['png_path'].name}")
    print(f"  → {result_ts3['n_stakeholders']} stakeholders, "
          f"{result_ts3['n_cities']} cities")
    print(f"  → {result_ts3['n_species']} top species across "
          f"{result_ts3['n_groups']} cities")
    print(f"  → Total tree mentions: {result_ts3['total_mentions']}")

# ── Chart 2d: ALL Tree Species × Cities ─────────────────────────────────────

print("\n[5b/6] Creating Tree Species heatmap — ALL species × Cities...")
result_ts4 = create_tree_species_heatmap(
    survey_df=survey_df,
    landscape_gdf=landscape_gdf,
    output_dir=viz_output_dir,
    top_n_species=None,
    group_by='city',
    show_plot=True,
)

if result_ts4.get('success'):
    print(f"\n✓ Chart 2d saved: {result_ts4['png_path'].name}")
    print(f"  → {result_ts4['n_stakeholders']} stakeholders, "
          f"{result_ts4['n_cities']} cities")
    print(f"  → All {result_ts4['n_species']} species across "
          f"{result_ts4['n_groups']} cities")
    print(f"  → Total tree mentions: {result_ts4['total_mentions']}")

print("\n" + "=" * 80)
print("Section 6.6 complete.")
print("=" * 80)

---

## Summary and Next Steps

This notebook produced a comprehensive, multi-method inventory of urban heat island hotspots for five Nigerian cities, with extended stakeholder analysis across all 28 workshop cities. The key outputs are:

1. **Enriched hotspot polygon dataset** — GeoDataFrame with 36+ attributes per polygon (detection method, period, contour level, landscape character, LST statistics), exported as GeoJSON in both WGS84 and Web Mercator projections.

2. **Stakeholder validation results** — Hit-rate analysis quantifying agreement between satellite-derived hotspots and stakeholder-reported heat-stress locations (Sections 5.2–5.3).

3. **Publication-ready figures** — Multi-scale visualization suite:
   - **Section 6.1:** Per-city grid maps (Day | Night | Persistent) with P90/Gi\* polygons on Cluster25 landscape base
   - **Section 6.2:** Gi\* Z-score distribution histograms with statistical annotations
   - **Section 6.3:** Stacked bar charts of hotspot area by contour level, method, and landscape composition
   - **Section 6.4:** Half-donut cross-section charts showing population exposure stratified by period (Day/Night), method (P90/Gi\*), and contour level (90%/75%/50%) — one per city
   - **Section 6.5:** National summary map showing all 28 cities on Nigeria's Cluster25 landscape with proportional symbols for Day/Night population exposure
   - **Section 6.6:** Stakeholder perception analysis by landscape character:
     - Reasons/mitigation chart (side-by-side stacked bars with city breakdown)
     - 4 tree species heatmaps (top-15/all × cluster25/city) showing unique stakeholder counts

All outputs are stored under `data/outputs/05_hotspot_analysis/`.

**Next:** Notebook 06 uses the enriched hotspot polygons and underlying LST rasters as inputs for LST modelling, investigating the relationship between land surface temperature and the physiographic setting of cities assembled in this analysis.